# 📗 GDS 중심성: PageRank 와 매개·근접 중심성

지난 시간에 투영을 만들었습니다. 이제 그 위에서 **"어느 노드가 가장 중요한가"** 를 계산합니다.

그런데 "중심"은 한 가지가 아닙니다. **선을 많이 가진 것**이 중심일 수도 있고, **중요한 것들과 이어진 것**이 중심일 수도 있고, **서로 다른 무리를 잇는 길목**이 중심일 수도 있습니다. 세 가지 중심성(지난 시간의 **차수**, 오늘의 **PageRank** 와 **매개 중심성**)이 이 세 가지 다른 질문에 하나씩 답합니다. 그리고 **같은 그래프에서 서로 다른 답을 냅니다.** 마지막에 근접 중심성을 하나 더 봅니다.

## 이 기술은 어디에 쓰일까요?

오늘 배우는 것은 전부 **그래프에서 순위를 매기는 일**입니다. 약·질병 그래프와 전철 노선은 연습 재료일 뿐이고, 같은 질문이 현장에서 매일 나옵니다.

| 현장의 질문 | 오늘의 도구 | 실제로 쓰이는 곳 |
|---|---|---|
| "전체에서 **가장 중요한** 것은" | PageRank (1절) | 구글 검색 순위(이 알고리즘의 고향), 논문·특허의 영향력, SNS 에서 실제로 영향력 있는 계정, 거래망에서 **돈이 모이는 계좌** 찾기 |
| "**나에게** 중요한 것은" | 개인화 PageRank (3절) | 트위터의 "팔로우 추천", 핀터레스트의 핀 추천, "이 약과 관련 깊은 질병", "이 고객이 다음에 살 상품" |
| "**끊기면 큰일 나는** 길목은" | 매개 중심성 (5절) | 통신망에서 죽으면 안 되는 장비, 공급망의 병목, 전염병을 막을 차단 지점, 조직에서 부서를 잇는 **연결 담당자** |
| "**어디로든 빨리 닿는** 자리는" | 근접 중심성 (6-2) | 물류 창고·응급 거점 **입지**, 소문이 가장 빨리 퍼지는 자리, 전철에서 어디든 갈아타기 편한 역 |

**이 과정에서 특히 중요한 이유.** 35일차부터 만드는 GraphRAG 는 질문을 받으면 지식 그래프에서 **후보를 잔뜩** 찾아 옵니다. 문제는 그다음입니다. 후보 100개 중 **무엇을 먼저 보여 줄 것인가.** 그 순서를 정하는 데 오늘의 PageRank 를 그대로 씁니다(35일차 커리큘럼에 **PageRank 재랭킹**이라는 이름으로 들어 있습니다). 질문에 나온 개체를 출발점으로 두고 **그 관점에서 관련 깊은 개체**를 찾는 것이 개인화 PageRank 이고, 그래프에서 여러 주제를 잇는 **길목 개체**를 찾는 것이 매개 중심성입니다. 오늘은 약과 역으로 연습하지만, 도구는 그대로 지식 그래프의 개체와 관계에 옮겨 갑니다.

## PageRank 란 무엇인가

오늘 다루는 세 중심성의 중심에 있는 알고리즘이므로 **정의부터** 확인합니다. 웹페이지 순위를 매기려고 고안된 것으로, "많이 참조되는 페이지가 참조한 페이지는 중요하다"는 발상을 그래프에 그대로 옮긴 것입니다.

> **한 노드의 점수는, 그 노드를 가리키는 이웃들이 나눠 준 몫을 모두 더한 값이다.**

식으로 쓰면 이렇습니다.

$$ PR(v) = (1 - d) \; + \; d \sum_{u \,\in\, In(v)} \frac{PR(u)}{L(u)} $$

| 기호 | 뜻 |
|---|---|
| $PR(v)$ | 구하려는 노드 $v$ 의 점수 |
| $In(v)$ | $v$ 를 **가리키는** 이웃들 |
| $L(u)$ | 이웃 $u$ 가 **내보내는 선의 개수** |
| $PR(u) / L(u)$ | 이웃 $u$ 가 $v$ 에게 넘기는 몫. 자기 점수를 **자기 선 개수로 나누어** 넘긴다 |
| $d$ | **감쇠 계수**(damping factor), 기본 `0.85`. 점수의 85% 는 선을 타고 흐르고 나머지는 모든 노드에 고르게 뿌린다 |

이 식에서 읽어 낼 것은 둘입니다.

1. **가리키는 이웃이 많을수록 점수가 높다.** 더할 항이 늘어나기 때문입니다. 여기까지는 차수와 같습니다.
2. **보내는 쪽의 선이 적을수록 받는 몫이 크다.** 선이 하나뿐인 이웃은 자기 점수 전부를 그 노드에 넘기고, 선이 200개인 이웃은 200분의 1만 넘깁니다. **차수와 갈리는 지점이 여기입니다.**

### 왜 반복 계산이 필요한가
우변에 $PR(u)$ 가 다시 들어 있습니다. 한 노드의 점수를 구하려면 이웃의 점수가 필요하고, 이웃의 점수를 구하려면 또 그 이웃의 점수가 필요합니다. **자기 자신을 참조하는 정의**라 한 번의 계산으로 풀리지 않습니다.

그래서 **모든 노드에 같은 초기값을 주고, 위 식을 반복해 대입**합니다. 값이 더 변하지 않는 상태를 **수렴했다**고 하고, 그때의 값이 답입니다. 몇 번까지 반복할지가 `maxIterations`, 변화가 얼마나 작아지면 멈출지가 `tolerance` 입니다(`maxIterations` 는 1-2 에서, `tolerance` 는 3절에서 직접 바꿔 봅니다).

> **초기값은 `0` 입니다**(GDS 기준. 교과서에 흔한 `1` 이나 `1/노드수` 가 아닙니다). 그래서 **첫 회는 계산이라기보다 씨앗 뿌리기**입니다. 합산 항이 전부 0 이라 모든 노드가 똑같이 $1 - d =$ `0.15` 를 받습니다. 실제로 `maxIterations: 1` 로 돌려 보면 노드 전부가 정확히 `0.15` 로 나옵니다. **둘째 회부터** 그 `0.15` 가 선을 타고 흐르기 시작하고, 그때 비로소 노드마다 값이 갈립니다.

### 점수를 읽는 법
$d = 0.85$ 이므로 **가리키는 이웃이 하나도 없는 노드**는 합산 항이 0 이 되어 점수가 $1 - d = $ **`0.15`** 로 정해집니다. 이 구현에서 나올 수 있는 **하한**입니다. 상한은 정해져 있지 않습니다. 오늘 보게 될 1위 점수 `8.08` 은 개수나 확률이 아니라 **상대적인 크기**이므로, 절대값을 외울 필요 없이 **순위로 읽습니다.**

## ⏪ 복습: 지난 시간까지

- **GDS 는 투영(메모리 사본) 위에서 계산합니다.** `gds.graph.project(이름, 레이블, 관계)`.
- **무엇을 담을지 고르는 것이 곧 질문을 정하는 일입니다.** 담지 않은 레이블·관계는 계산에 끼지 못합니다.
- **`orientation: 'UNDIRECTED'`** 로 방향을 지울 수 있고, 그러면 관계가 2배로 담깁니다. 이 한 줄을 빠뜨리면 답이 조용히 틀립니다(아래 "왜 무방향으로 담나요?" 에서 확인합니다).
- 투영은 **시점 사본**이라 원본 변경을 따라가지 않습니다.
- **차수(degree)**: `gds.degree.stream` 으로 선의 개수를 세었습니다. `orientation` 으로 나가는·들어오는·양쪽을, `relationshipTypes` 로 관계 종류를 골랐습니다. 오늘은 그 차수를 **견줄 기준으로 두고** 더 나아갑니다.

**오늘의 목표**

**1. PageRank**
- [ ] (1-1) **PageRank** 가 차수와 무엇이 다른지 설명하고 `gds.pageRank.stream` 으로 계산한다.
- [ ] (1-2) PageRank 의 **설정 셋**(`dampingFactor`·`maxIterations`·`tolerance`)의 뜻을 알고, `maxIterations` 를 바꿔 순위가 어디까지 흔들리는지 본다.

**2. 실행 모드**
- [ ] (2-1) **실행 모드**(`stream`·`stats`·`mutate`·`write`)를 구분하고 `write` 로 점수를 저장한다.

**3. 개인화 PageRank**
- [ ] (3-1) **개인화 PageRank**(`sourceNodes`)로 출발점을 정한 관점의 순위를 얻는다.

**4. 범위 필터**
- [ ] (4-1) **`nodeLabels`** 로 노드 종류를 좁히고, 지난 시간의 `relationshipTypes` 와 함께 써 본다.

**5. 매개 중심성**
- [ ] (5-1) **매개 중심성(betweenness)** 으로 길목을 찾는다.

**6. 세 잣대 비교**
- [ ] (6-1) 세 중심성의 **상위 10 이 서로 얼마나 다른지** 직접 세어 본다.
- [ ] (6-2) **근접 중심성**을 하나 더 보고, 새 잣대가 값어치가 있는지 겹침으로 따진다.

아래 준비 셀들을 위에서부터 실행하세요. **연결 → 초기화(투영·그래프) → 데이터 적재 두 번 → 오늘 쓸 투영 만들기** 순서입니다. 적재가 두 번인 이유는 오늘 **서로 다른 두 그래프**를 쓰기 때문입니다.

In [ ]:
# [제공 코드] Neo4j 연결: 이 셀은 실행만 하세요(내용은 이해하지 않아도 됩니다).
# - .env 의 NEO4J_URI/NEO4J_USER/NEO4J_PASSWORD 로 데이터베이스에 연결합니다.
# - run_cypher("쿼리", 파라미터=값) 가 결과를 dict 리스트로 돌려줍니다. 이 헬퍼로 Cypher 를 실행합니다.
# - 반드시 "실습 전용" 데이터베이스에 연결하세요. 아래 실습이 그래프를 지우고 새로 만듭니다.
import os

from dotenv import load_dotenv
from neo4j import GraphDatabase

# 1) 접속 정보 읽기: .env 의 키=값을 환경변수로 올려 둔다(비밀번호를 코드에 적지 않으려고)
load_dotenv(".env")       # 같은 폴더의 .env
load_dotenv("../.env")    # 정답 폴더에서 실행하는 경우

# os.getenv(키, 기본값): .env 를 못 읽어도 에러가 아니라 이 기본값으로 조용히 넘어간다.
# 그러니 이 셀 마지막 줄에 찍히는 주소가 "실습 전용 DB" 가 맞는지 눈으로 꼭 확인한다
NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "neo4j")

# 2) 드라이버: 노트북과 데이터베이스를 잇는 통로를 하나 열어 둔다(노트북이 끝날 때까지 재사용)
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()   # 지금 바로 접속을 시험한다. 여기서 에러가 나면 주소나 계정이 틀린 것


# 3) 공용 헬퍼: 앞으로 모든 Cypher 는 이 함수 하나로 실행한다
def run_cypher(query, **params):
    """Cypher 실행 -> 결과를 dict 리스트로 반환(수업 공용 헬퍼)."""
    # session 은 쿼리를 실행하는 단위. with 블록을 벗어나면 알아서 닫힌다
    with driver.session() as session:
        # record.data() 가 한 행을 dict 로 바꾼다. 그 키는 RETURN 에 적은 별칭이 된다
        return [record.data() for record in session.run(query, **params)]


print("Neo4j 연결:", NEO4J_URI)

In [ ]:
# [제공 코드] 실습 전용 DB 초기화: 지우기 전에 이 DB 가 맞는지 먼저 확인합니다.
UNIT_LABELS = ['Compound', 'DayType', 'Disease', 'ExDenseEvent', 'ExDisease', 'ExDomestic', 'ExDrug', 'ExEvent', 'ExForeign', 'ExIdKey', 'ExKing', 'ExKinng', 'ExNameKey', 'ExNodeKeyDemo', 'ExPerson', 'ExReign', 'ExScopeEvent', 'ExThrone', 'ExUniqueDemo', 'ExWorld', 'ExYear', 'FlatDrug', 'FlatKing', 'FlatOrder', 'Gene', 'IdKey', 'Line', 'NameKey', 'NodeClass', 'NodeDay', 'NodeDrug', 'NodeKing', 'NodeMonth', 'NodeOrder', 'NodeYear', 'PharmacologicClass', 'Station', 'SurveyDay', 'SurveyYear', 'Symptom', 'TempStation', 'TryDay', 'TryLine', 'TryStation']   # 이 단원이 만드는 레이블 전부(앞 일차가 남긴 것까지)

# 이 단원 것이 아닌 노드가 하나라도 있으면 지우지 않고 멈춥니다.
# .env 의 주소가 어긋나도 접속은 조용히 성공하므로, 지우기 전에 확인하는 수밖에 없습니다.
# all 로 보는 이유: any 로 보면 :Person:PatientRecord 처럼 한 레이블만 겹치는 남의 노드가 통과합니다
foreign = run_cypher("""
MATCH (n) WHERE size(labels(n)) = 0
   OR NOT all(label IN labels(n) WHERE label IN $unit_labels)
RETURN DISTINCT labels(n) AS labels LIMIT 5""", unit_labels=UNIT_LABELS)
if foreign:
    raise RuntimeError(
        f"{NEO4J_URI} 에 이 단원 것이 아닌 노드가 있습니다: {foreign}\n"
        "다른 실습이나 개인 데이터가 든 DB 로 보여 초기화를 멈췄습니다.\n"
        ".env 의 NEO4J_URI 가 실습 전용 DB 를 가리키는지 먼저 확인하세요.\n"
        "주소가 맞다면 위 레이블은 앞 실습이 남긴 것입니다. UNIT_LABELS 에 더하고 다시 실행하세요.")

# 여기까지 왔으면 이 DB 에는 이 단원이 만든 노드밖에 없습니다.
# 1) 메모리에 올라온 투영부터 내린다. 투영은 이름이 겹치면 다시 못 만들어 재실행이 막힌다
for _g in run_cypher("CALL gds.graph.list() YIELD graphName RETURN graphName"):
    run_cypher("CALL gds.graph.drop($name) YIELD graphName RETURN graphName",
               name=_g["graphName"])

# 2) 저장된 그래프를 지운다. DETACH: 노드에 붙은 관계까지 함께 지운다
run_cypher("MATCH (n) DETACH DELETE n")

print("초기화 완료:", NEO4J_URI,
      "· 남은 노드:", run_cypher("MATCH (n) RETURN count(n) AS n")[0]["n"],
      "· 남은 투영:", len(run_cypher("CALL gds.graph.list() YIELD graphName RETURN graphName")))

In [ ]:
# [제공 코드] 의료 지식 그래프 적재: 이 셀은 실행만 하세요(2초쯤 걸립니다).
# data/hetionet_*.csv 는 Hetionet v1.0 에서 재배포 가능한 출처(CC0/CC BY)만 골라 낸 조각입니다.
from pathlib import Path

import pandas as pd

DATA_DIR = Path("data") if Path("data").exists() else Path("../data")   # 정답 폴더에서도 돌게
NODE_LABELS = ["Compound", "Disease", "Gene", "Symptom", "PharmacologicClass"]
# 관계 타입마다 양끝 레이블이 정해져 있습니다. 적재할 때 이 표로 레이블을 찍어 줘야
# MATCH 가 인덱스를 타고, 그래야 9만 건이 몇 초 안에 들어갑니다.
REL_ENDS = {
    "TREATS": ("Compound", "Disease"),
    "PALLIATES": ("Compound", "Disease"),
    "BINDS": ("Compound", "Gene"),
    "UPREGULATES_CG": ("Compound", "Gene"),
    "DOWNREGULATES_CG": ("Compound", "Gene"),
    "ASSOCIATES": ("Disease", "Gene"),
    "UPREGULATES_DG": ("Disease", "Gene"),
    "DOWNREGULATES_DG": ("Disease", "Gene"),
    "RESEMBLES_DD": ("Disease", "Disease"),
    "RESEMBLES_CC": ("Compound", "Compound"),
    "PRESENTS": ("Disease", "Symptom"),
    "INCLUDES": ("PharmacologicClass", "Compound"),
}

# 1) 레이블마다 id 인덱스를 먼저 만든다. 관계를 붙일 때 이 인덱스로 노드를 찾는다
for _label in NODE_LABELS:
    run_cypher(f"CREATE INDEX {_label.lower()}_id IF NOT EXISTS FOR (n:{_label}) ON (n.id)")

# 2) 노드 csv 를 읽어 레이블별로 나눠 담는다. 레이블마다 CREATE 쿼리가 달라서 미리 갈라 둔다
_nodes = {_label: [] for _label in NODE_LABELS}
for _row in pd.read_csv(DATA_DIR / "hetionet_nodes.csv").to_dict("records"):
    _nodes[_row["label"]].append({"id": _row["id"], "name": _row["name"]})
for _label, _rows in _nodes.items():
    # 초기화 직후라 같은 노드가 있을 수 없다. MERGE 대신 CREATE 가 훨씬 빠르다
    run_cypher(f"UNWIND $rows AS row CREATE (n:{_label}) SET n.id = row.id, n.name = row.name",
               rows=_rows)

# 3) 관계 csv 도 타입별로 나눠 담는다
_edges = {_rel: [] for _rel in REL_ENDS}
for _row in pd.read_csv(DATA_DIR / "hetionet_edges.csv").to_dict("records"):
    _edges[_row["rel"]].append({"s": _row["source"], "t": _row["target"]})
for _rel, _rows in _edges.items():
    _src, _dst = REL_ENDS[_rel]   # 이 타입의 출발·도착 레이블을 위 표에서 꺼낸다
    # 2만 건씩 끊어 보낸다. 9만 건을 한 트랜잭션에 넣으면 메모리가 크게 뛴다
    for _start in range(0, len(_rows), 20000):
        run_cypher(f"UNWIND $rows AS row "
                   f"MATCH (a:{_src} {{id: row.s}}), (b:{_dst} {{id: row.t}}) "
                   f"CREATE (a)-[:{_rel}]->(b)", rows=_rows[_start:_start + 20000])

print("노드:", run_cypher("MATCH (n) RETURN count(n) AS c")[0]["c"],
      "/ 관계:", run_cypher("MATCH ()-[r]->() RETURN count(r) AS c")[0]["c"])

### 따라하기에 쓸 그래프: 수도권 전철

시연을 그대로 따라 치면 손은 움직여도 판단은 늘지 않습니다. 그래서 오늘 따라하기는 **도메인이 다른 그래프**로 합니다. 배운 잣대가 데이터의 성격과 상관없이 드는 도구인지 거기서 확인하게 됩니다.

모델은 두 가지만 보면 됩니다. **역과 역을 잇는 구간**, 그리고 **역이 속한 노선**입니다. 다음 셀이 이 그래프를 적재합니다.

<img src="images/전철그래프_모델.png" width="860">

In [ ]:
# [제공 코드] 수도권 전철 그래프 적재: 이 셀은 실행만 하세요(1초쯤 걸립니다).
# data/seoul_subway_*.csv 는 30일차에서 쓴 것과 같은 파일입니다.
# 자료 출처: OpenStreetMap contributors (ODbL).
#   (:Station {name})-[:NEXT_TO {line, km}]->(:Station)   이웃한 두 역
#   (:Station)-[:ON_LINE]->(:Line {name})                 역이 속한 노선
# 앞의 의료 그래프와는 선이 하나도 이어지지 않습니다. 한 DB 에 별개의 그래프 둘이 있는 셈입니다.
import csv
from pathlib import Path

# 앞의 의료 적재 셀에서 이미 정했지만, 이 셀만 따로 돌려도 되도록 여기서 한 번 더 정한다
DATA_DIR = Path("data") if Path("data").exists() else Path("../data")


def _read_subway(filename):
    """전철 CSV 한 장을 dict 목록으로 읽는다. 값은 전부 문자열이라 드라이버에 그대로 넘어간다."""
    with open(DATA_DIR / filename, encoding="utf-8") as f:
        return list(csv.DictReader(f))


_st = _read_subway("seoul_subway_stations.csv")

# 이름으로 찾을 일이 많으니 인덱스부터 만든다(적재 속도가 여기서 갈린다)
run_cypher("CREATE INDEX station_name IF NOT EXISTS FOR (n:Station) ON (n.name)")
run_cypher("CREATE INDEX line_name IF NOT EXISTS FOR (n:Line) ON (n.name)")

# 1) 역 노드. 초기화 직후라 같은 이름이 있을 수 없어 MERGE 대신 CREATE 가 빠르다
run_cypher("UNWIND $rows AS row CREATE (:Station {name: row.name})",
           rows=[{"name": _r["name"]} for _r in _st])

# 2) 노선 노드. 역마다 적힌 노선 이름을 모아 중복을 없앤다
_lines = sorted({_ln for _r in _st for _ln in _r["lines"].split("|")})
run_cypher("UNWIND $rows AS name CREATE (:Line {name: name})", rows=_lines)

# 3) 역-노선 소속. 한 역이 여러 노선에 속하면 그만큼 선이 생긴다(그게 환승역이다)
run_cypher("UNWIND $rows AS row "
           "MATCH (s:Station {name: row.s}), (l:Line {name: row.l}) "
           "CREATE (s)-[:ON_LINE]->(l)",
           rows=[{"s": _r["name"], "l": _ln} for _r in _st for _ln in _r["lines"].split("|")])

# 4) 이웃 구간. 두 노선이 같은 구간을 함께 지나는 곳이 있어 그때는 선이 두 번 생긴다
run_cypher("UNWIND $rows AS row "
           "MATCH (a:Station {name: row.f}), (b:Station {name: row.t}) "
           "CREATE (a)-[:NEXT_TO {line: row.line, km: toFloat(row.km)}]->(b)",
           rows=[{"f": _r["from"], "t": _r["to"], "line": _r["line"], "km": _r["km"]}
                 for _r in _read_subway("seoul_subway_edges.csv")])

print("역:", run_cypher("MATCH (n:Station) RETURN count(n) AS c")[0]["c"],
      "/ 노선:", run_cypher("MATCH (n:Line) RETURN count(n) AS c")[0]["c"],
      "/ 구간:", run_cypher("MATCH ()-[r:NEXT_TO]->() RETURN count(r) AS c")[0]["c"],
      "/ 소속:", run_cypher("MATCH ()-[r:ON_LINE]->() RETURN count(r) AS c")[0]["c"])

## 오늘 쓰는 투영

한 데이터베이스에 이어지지 않은 그래프 둘이 들어 있어도 계산이 섞이지 않습니다. **투영이 담는 것만 계산에 끼기** 때문입니다. 담을 것을 고르는 것으로 원하는 쪽만 꺼내 씁니다.

시연은 의료 그래프를 **두 가지로 투영**해 쓰고, 따라하기는 전철 그래프를 씁니다. 두 그래프의 규모는 방금 적재 셀이 찍어 준 그대로입니다. 셋 중 둘을 지금 만들고, 나머지 하나는 필요해지는 자리에서 만듭니다.

| 투영 | 담은 것 | 묻는 질문 | 언제 |
|---|---|---|---|
| **`drugGraph`** | 약물·질병·약효분류와 그 사이 관계 5종 | 이 그래프에서 **무엇이 중심인가** | 지금 |
| **`subwayGraph`** | 역(`Station`)과 이웃 구간 `NEXT_TO` | **어느 역이 중심인가**(따라하기 전용) | 지금 |
| **`fullGraph`** | 레이블 5종·관계 12종 **전부** | 이 약과 **이어진 것은 무엇인가** | 3절 |

`drugGraph` 가 유전자를 빼는 이유는 3절에서 밝힙니다. 지금은 "약과 병 이야기만 담은 조각"이라고 생각하세요. `subwayGraph` 가 `Line` 을 빼는 것도 같은 성격의 결정입니다. 알고 싶은 것이 **어느 역이 중심인가**라서 노선을 뺐습니다.

무엇을 담고 무엇을 뺐는지, 빼면 무엇이 달라지는지는 아래 두 그림에 있습니다.

<img src="images/drugGraph_구성.png" width="900">

<img src="images/subwayGraph_구성.png" width="900">

세 번째 `fullGraph` 는 3절에서 만듭니다. 무엇이 더 들어오는지 미리 보아 두면 앞의 두 조각이 **무엇을 빼고 있는지**가 분명해집니다.

<img src="images/fullGraph_구성.png" width="900">

In [ ]:
# [제공 코드] 오늘 쓸 투영 두 개 만들기: 실행만 하세요(문법은 지난 시간에 배웠습니다).
# drugGraph: 약물을 중심에 둔 조각. 약물·질병·약효분류 노드와 그 사이 관계 5종만 담는다
# 관계는 전부 무방향으로 담는다. 방향이 남으면 들어오는 선이 없는 종류가 통째로 바닥값에 붙는다(바로 아래에서 확인한다)
run_cypher("""
    CALL gds.graph.project('drugGraph',
        ['Compound', 'Disease', 'PharmacologicClass'],   // 담을 노드 종류. 여기 없는 Gene 은 계산에 못 낀다
        {TREATS:       {orientation: 'UNDIRECTED'},      // 관계 타입마다 방향 설정을 따로 적는다
         PALLIATES:    {orientation: 'UNDIRECTED'},
         INCLUDES:     {orientation: 'UNDIRECTED'},
         RESEMBLES_DD: {orientation: 'UNDIRECTED'},
         RESEMBLES_CC: {orientation: 'UNDIRECTED'}})     // 하나라도 빠뜨리면 그 타입만 방향이 남는다
    YIELD graphName RETURN graphName
""")
# subwayGraph: 수도권 전철 조각. 따라하기에서만 씁니다
# 역과 이웃 구간만 담고 Line 노드는 뺍니다. 노선을 함께 담으면 역을 잔뜩 거느린 노선 노드가
# 어느 잣대로 재도 1위를 차지해 순위표를 통째로 삼킵니다(무엇을 담을지가 곧 질문입니다)
run_cypher("""
    CALL gds.graph.project('subwayGraph',
        ['Station'],                                     // 역만 담는다. Line 은 일부러 뺀다
        {NEXT_TO: {orientation: 'UNDIRECTED'}})          // 이웃한 두 역. 열차는 양쪽으로 다닌다
    YIELD graphName RETURN graphName
""")
# 두 투영이 제대로 올라왔는지 크기로 확인한다. gds.graph.list 는 지금 메모리에 올라온 투영을 준다
for _row in run_cypher("CALL gds.graph.list() YIELD graphName, nodeCount, relationshipCount "
                       "RETURN graphName, nodeCount, relationshipCount ORDER BY graphName"):
    print(f"  {_row['graphName']:14} 노드 {_row['nodeCount']:>6,}  관계 {_row['relationshipCount']:>7,}")

# 둘 다 무방향이라 관계 수가 원본의 2배다(drugGraph 원본 9,203 · subwayGraph 원본 778)

### 왜 무방향으로 담나요?

투영 설정에 `orientation: 'UNDIRECTED'` 가 관계마다 붙어 있습니다. **PageRank 는 원래 방향 그래프에 쓰는 알고리즘**이니 이상해 보일 수 있습니다. 웹에서 "A 페이지가 B 를 링크했다"는 A 가 B 를 **지목했다**는 뜻이고 그 반대는 성립하지 않죠. 그래서 웹 링크·논문 인용·팔로우처럼 **화살표 자체가 뜻을 갖는** 그래프에서는 방향을 그대로 둡니다.

이 그래프는 다릅니다. `(:Compound)-[:TREATS]->(:Disease)` 의 화살표는 지목이 아니라 **어느 쪽이 약이고 어느 쪽이 병인지 적어 둔 표기**입니다. 그대로 두고 계산하면 답이 무너집니다. 방향을 살린 채 같은 조각을 재 보면 이렇습니다.

| 방향을 그대로 둔 drugGraph | PageRank 결과 |
|---|---|
| 약효분류 345개 | **전부 `0.15`**. 1-d 바닥값에 붙어 모두 동점 |
| 약물 1,531개 | 평균 `0.34` |
| 질병 136개 | 평균 `1.99`, 최대 `14.34`. **상위 10 이 전부 질병** |

`INCLUDES` 는 약효분류에서 약물로 **나가기만** 합니다. 그래서 약효분류로 **들어오는 선이 하나도 없고**, 받을 몫이 없으니 345개가 전부 바닥값에 붙습니다. 줄을 세우라고 돌린 계산인데 그 종류 전체가 동점이 되는 것이죠. `TREATS`·`PALLIATES` 도 약물에서 질병으로만 가니 질병 쪽에는 점수가 쌓이기만 합니다.

**무방향은 기본값이 아니라 이 데이터에 맞춰 고른 것입니다.** 판단 기준은 하나입니다.

- 화살표가 **지목·추천·흐름**을 뜻하면(웹 링크, 논문 인용, 팔로우, 송금) 방향을 살립니다.
- 화살표가 **양끝의 종류를 적어 둔 표기**이거나 관계가 서로 대등하면(약과 병의 연관, 화학적 닮음, 함께 등장) 무방향으로 담습니다.

> **에러가 나지 않는 것이 이 함정의 핵심입니다.** 방향을 남겨도 투영은 만들어지고 알고리즘도 돌고 결과도 나옵니다. 위 표처럼 검산해 보지 않으면 틀린 줄 모르고 씁니다.

---
# 1. PageRank: 이웃의 점수까지 본다

지난 시간 마지막에 **차수**로 "선이 몇 개인가"를 세었습니다. 여기서는 `gds.pageRank.stream` 으로 **선이 어디에서 왔는지**까지 보는 순위를 매기고, 그 차수 상위 10 과 나란히 놓아 **몇 개나 겹치는지** 직접 셉니다. 이어서 설정 셋(`dampingFactor`·`maxIterations`·`tolerance`)이 순위의 어느 자리를 흔드는지 봅니다.

먼저 견줄 기준을 만듭니다. 지난 시간에 배운 `gds.degree.stream` 을 오늘 조각(`drugGraph`)에 그대로 돌립니다. `drugGraph` 는 무방향 투영이라 `orientation` 을 줄 필요가 없습니다. 오히려 주면 안 됩니다. `'UNDIRECTED'` 를 한 번 더 주면 이미 양방향으로 담긴 관계를 다시 두 번 세어 차수가 2배로 부풀고, 에러는 나지 않습니다(지난 시간 6-2 에서 본 함정입니다).

In [ ]:
# [제공 코드] 견줄 기준: 지난 시간에 배운 차수를 오늘 조각에 그대로 돌린다
deg_rows = run_cypher("""
    CALL gds.degree.stream('drugGraph')
    YIELD nodeId, score                            // GDS 는 노드가 아니라 내부 id 와 점수만 내준다
    RETURN gds.util.asNode(nodeId).name AS name,   // 그 id 로 원본 노드를 되찾아 이름을 꺼낸다
           labels(gds.util.asNode(nodeId))[0] AS kind,
           score AS degree                         // degree 에서는 score 가 곧 선의 개수다
    ORDER BY degree DESC, name                     // 동점이면 이름순. 없으면 실행마다 순서가 흔들린다
    LIMIT 10
""")
for rank, row in enumerate(deg_rows, 1):
    print(f"{rank:2}. {row['name']:26} {row['kind']:20} {int(row['degree']):>4}")

## 1-1. PageRank 로 다시 재고 차수와 견주기

### 왜 필요할까요?
차수는 이웃을 **세기만** 합니다. 맨 앞에서 본 정의가 이 지점을 보완합니다. 이웃이 넘겨 주는 몫은 $PR(u)/L(u)$ 였습니다. **누가 넘기는가**(그 이웃의 점수)와 **몇 갈래로 나누어 넘기는가**(그 이웃의 선 개수)가 함께 반영됩니다. 차수는 이 몫을 모두 1 로 두고 세는 것과 같습니다.

<img src="images/차수_대_pagerank.png" width="860">

### 문법

```cypher
CALL gds.pageRank.stream('투영이름') YIELD nodeId, score
```

차수와 호출 모양이 같습니다. **`score` 의 뜻만 다릅니다.** PageRank 의 점수는 개수가 아니라 **상대적인 크기**라 절대값을 외울 필요가 없습니다. 순위만 보면 됩니다.

In [ ]:
# 같은 투영, 다른 잣대. 이번엔 이웃의 점수까지 반영해 순위를 매긴다
pr_rows = run_cypher("""
    CALL gds.pageRank.stream('drugGraph')
    YIELD nodeId, score          // 앞 셀과 프로시저 이름만 다르다. score 의 뜻이 개수에서 상대적 크기로 바뀐다
    RETURN gds.util.asNode(nodeId).name AS name,
           labels(gds.util.asNode(nodeId))[0] AS kind,
           score
    ORDER BY score DESC, name    // 여기서도 동점 대비로 이름을 둘째 기준에 둔다
    LIMIT 10
""")
for rank, row in enumerate(pr_rows, 1):
    print(f"{rank:2}. {row['name']:26} {row['kind']:20} {row['score']:.3f}")

### 두 순위를 나란히 놓고 세어 봅니다

말로 "다르다"고 하는 것보다 **몇 개나 겹치는지 세는 편**이 확실합니다. pandas 로 두 상위 10 을 붙여 봅니다.

In [ ]:
import pandas as pd

# 두 순위를 같은 길이의 열로 만들어 나란히 본다. 순위표는 이렇게 봐야 차이가 눈에 들어온다
compare = pd.DataFrame({
    "차수 Top10": [row["name"] for row in deg_rows],
    "PageRank Top10": [row["name"] for row in pr_rows],
})
compare.index = range(1, len(compare) + 1)   # 인덱스를 1부터 매겨 순위로 읽히게 한다
display(compare)

In [ ]:
# 집합으로 바꿔 교집합을 세면 '몇 개나 같은가'가 바로 나온다
# 겹치는 이름이 곧 '어느 잣대로 보든 중요한 것' 이다
shared = {row["name"] for row in deg_rows} & {row["name"] for row in pr_rows}
print("두 상위 10 에 함께 있는 노드:", len(shared), "개 ->", sorted(shared))

**3개**뿐입니다. 열 자리 중 일곱 자리가 바뀌었습니다. 두 잣대가 서로 다른 것을 재고 있다는 뜻입니다.

가장 크게 벌어진 노드를 봅니다. **Eltrombopag** 는 PageRank 4위인데 차수는 **469위**입니다. 반대로 차수 2위였던 `Diphenhydramine` 는 PageRank **10위**로 내려갔습니다. 다만 이 순위는 **기본 설정 기준**입니다. 반복을 더 돌리면 한 계단 올라올 만큼 아슬아슬한 자리인데, 조금 아래에서 직접 확인합니다.

In [ ]:
# PageRank 상위인데 차수는 낮은 노드의 이웃을 본다. 무엇과 이어져 있길래 점수가 올라갔나
drug_name = "Eltrombopag"
rows = run_cypher("""
    MATCH (c:Compound {name: $name})-[r]-(other)  // 화살표 없이 -[r]- 라 나가는 선·들어오는 선을 다 본다
    WHERE type(r) IN ['TREATS', 'PALLIATES', 'INCLUDES', 'RESEMBLES_CC', 'RESEMBLES_DD']  // drugGraph 에 담은 관계만 본다. 안 담은 관계를 세면 투영과 딴 답이 나온다
    RETURN type(r) AS rel_type, labels(other)[0] AS kind, other.name AS neighbor
    ORDER BY rel_type, neighbor
""", name=drug_name)
# other 는 GDS 결과가 아니라 실제 DB 노드라 labels(other) 로 레이블을 바로 꺼낼 수 있다
# 제목과 목록을 한 번에 찍어, 어느 약의 이웃인지 출력만 보고도 알게 한다
print(f"{drug_name} 의 이웃 {len(rows)}개\n" +
      "\n".join(f"  {row['rel_type']:12} {row['kind']:20} {row['neighbor']}"
                for row in rows))

In [ ]:
# 이웃이 몇 개인지보다, 그 이웃이 자기 점수를 몇 갈래로 나누어 넘기는지가 중요하다.
# 이웃이 넘겨 주는 몫 = 그 이웃의 점수 / 그 이웃의 선 개수. 그래서 이웃의 차수를 함께 센다
rows = run_cypher("""
    MATCH (c:Compound {name: $name})-[:INCLUDES]-(p:PharmacologicClass)  // 이 약이 속한 약효분류를 찾고
    RETURN p.name AS 약효분류, COUNT { (p)-[:INCLUDES]-() } AS 거느린약  // 그 분류가 거느린 약 수를 센다. 이것이 식의 L(u) 다
    ORDER BY 거느린약 DESC, 약효분류
""", name=drug_name)
print(f"{drug_name} 가 속한 약효분류 수:", len(rows),
      "· 그 분류들이 거느린 약 수:", [r["거느린약"] for r in rows])

이웃이 전부 **약효분류(`PharmacologicClass`)** 입니다. 그런데 그 분류들이 거느린 약 수를 보면 대부분 **1** 입니다. 이 약 하나만 들어 있는 분류라는 뜻입니다.

**이것이 점수가 오른 이유입니다.** PageRank 에서 이웃이 넘겨 주는 몫은 **그 이웃의 점수를 그 이웃의 선 개수로 나눈 값**입니다. 선이 하나뿐인 이웃은 나눌 곳이 없어 **자기 점수 전부**를 이 약에 넘깁니다. 반대로 수백 개와 이어진 이웃은 같은 점수를 수백 갈래로 나눠 넘기므로 한 갈래의 몫이 아주 작습니다.

> **정리하면**: 차수는 "몇 개와 이어졌나", PageRank 는 "**어떤 이웃**과 이어졌나"를 봅니다. 그 "어떤 이웃"을 정하는 것은 두 가지입니다. **그 이웃의 점수가 큰가**, 그리고 **그 이웃이 자기 점수를 몇 갈래로 나누어 넘기는가**.

## 1-2. 설정 셋: 얼마나 흘리고 몇 번 돌릴 것인가

PageRank 는 점수를 **여러 번 돌려 가며** 쌓는 알고리즘입니다. 그래서 돌리는 방식을 정하는 설정이 있고, 그중 셋을 봅니다. 두 번째 인자에 맵으로 넣습니다.

| 설정 | 뜻 | 기본값 |
|---|---|---|
| `dampingFactor` | 점수가 이웃으로 흘러가는 비율. 나머지는 모든 노드에 고르게 뿌린다 | `0.85` |
| `maxIterations` | 최대 몇 번 돌릴지 | `20` |
| `tolerance` | 점수 변화가 이보다 작아지면 그만 돈다 | `0.0000001` |

여기서 중요한 것은 **기본값 20회가 '수렴할 때까지'가 아니라는 점**입니다. 20번 돌고 그냥 멈춥니다. `.stats` 모드는 점수 대신 요약만 주는데, 거기에 **몇 번 돌았고 수렴했는지**가 들어 있습니다.

In [ ]:
# 반복 상한을 바꿔 가며 실제로 몇 번 돌고 멈추는지 본다. stats 는 점수 대신 요약만 준다
# didConverge 가 False 면 '수렴해서 멈춘 것이 아니라 상한에 걸려 멈춘 것'이다
for limit in [20, 50, 100]:
    row = run_cypher("CALL gds.pageRank.stats('drugGraph', {maxIterations: $n}) "
                     "YIELD ranIterations, didConverge "
                     "RETURN ranIterations, didConverge", n=limit)[0]
    print(f"  상한 {limit:3}회 -> 실제 {row['ranIterations']:2}회, 수렴 {row['didConverge']}")


기본값으로 돌린 앞의 순위표는 **아직 수렴하지 않은 상태**의 값이었습니다. 그래도 상위권은 웬만해선 그대로입니다. 문제는 점수가 아슬아슬하게 붙은 자리죠. 직접 확인해 봅니다.

In [ ]:
# 반복을 더 돌린 순위를 앞의 기본값 결과와 나란히 놓는다. 어느 자리가 흔들리는지 보려는 것이다
more = run_cypher("""
    CALL gds.pageRank.stream('drugGraph', {maxIterations: 50})  // 기본 20회 대신 50회를 돌린다
    YIELD nodeId, score
    RETURN gds.util.asNode(nodeId).name AS name
    ORDER BY score DESC, name
    LIMIT 10
""")
iter_compare = pd.DataFrame({
    "기본 20회": [row["name"] for row in pr_rows],   # 앞 절에서 기본값으로 뽑아 둔 목록
    "50회": [row["name"] for row in more],
})
iter_compare.index = range(1, len(iter_compare) + 1)
display(iter_compare)

1위부터 8위까지는 꿈쩍도 하지 않았는데 **9위와 10위가 자리를 바꿨습니다.** 앞의 순위표에서 두 점수는 `3.635` 와 `3.584` 로 1.4% 차이였습니다. 반복을 더 돌리자 그 좁은 간격이 뒤집힌 것입니다.

그래서 **순위를 인용할 때는 설정을 함께 적어야** 합니다. 다른 도구로 같은 그래프를 재서 순위가 조금 다르게 나왔다면, 데이터가 틀린 것이 아니라 반복 횟수가 달랐을 가능성을 먼저 봐야 합니다. 위 셀의 `{maxIterations: 50}` 자리에 `{dampingFactor: 0.5}` 를 대신 넣어 돌려 보면 흔들림이 더 큽니다. 점수가 멀리 못 가서 4위였던 `Eltrombopag` 가 3위로 올라옵니다.

### 🖐️ 함께 따라하기: 전철 그래프의 PageRank

여기서부터 따라하기는 **전철 그래프**로 합니다. 시연에서 본 것과 도메인이 다르니, 잣대가 정말 일반적인 도구인지 여기서 확인하는 셈입니다.

`subwayGraph` 에서 PageRank 상위 5개를 뽑고, 같은 투영의 **차수 상위 5개와 나란히** 출력해 비교하세요.

이번에는 쿼리를 그대로 쓰지 말고 **`top5(algo, graph_name)` 함수로 감싸세요.** 알고리즘 이름만 갈아 끼우면 되는 모양이라, 뒤의 5절과 응용에서 그대로 다시 씁니다.

**확인 기준**: 1위는 두 잣대 모두 `김포공항` 으로 같지만, **2위부터 갈립니다** (차수 2위 `공덕` / PageRank 2위 `대곡`).

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) top5(algo, graph_name) 함수를 만든다 - 상위 5개 노드 이름을 리스트로 돌려준다
#    쿼리 안의 gds.{algo}.stream 은 f-string 으로 갈아 끼우고, 투영 이름은 파라미터 $g 로 넘긴다
# 2) top5 를 'degree' 와 'pageRank' 로 두 번 불러 두 목록을 만든다
# 3) 두 목록을 pandas DataFrame 두 열로 만들어 display 한다

1위는 두 잣대 모두 `김포공항` 입니다. 노선이 5개 지나는 역이라 어느 잣대로 재도 앞자리를 내주지 않습니다.

갈리는 것은 2위부터입니다. 차수로는 `공덕`·`대곡`·`왕십리` **셋이 모두 8** 이라 이름순으로 줄을 섰을 뿐인데, PageRank 는 그중 `대곡` 과 `왕십리` 를 `공덕` 위로 올려 놓습니다. **이웃이 몇 개인지가 아니라 어떤 이웃인지**가 반영되기 때문입니다. 같은 개수라도 그 이웃이 자기 점수를 몇 갈래로 나누어 넘기느냐가 다릅니다.

### ✅ 바로 확인 퀴즈

**1.** PageRank 점수가 `8.08` 이라는 것은 무슨 뜻인가요?

<details><summary>정답 보기</summary>

**상대적인 크기**일 뿐 개수나 확률이 아닙니다. 다른 노드와 비교해 얼마나 높은지만 의미가 있으므로 **순위로 읽습니다.**

</details>

**2.** 선이 13개뿐인 노드가 선이 63개인 노드보다 PageRank 가 높을 수 있는 이유는?

<details><summary>정답 보기</summary>

PageRank 는 **이웃이 넘겨 주는 몫**까지 보기 때문입니다. 그 몫은 이웃의 점수를 **이웃의 선 개수로 나눈 값**이라, 선이 적은 이웃과 이어진 쪽은 한 이웃에게서 큰 몫을 받습니다. 선이 많아도 그 이웃들이 저마다 수백 갈래로 쪼개 넘기면 합이 작을 수 있습니다.

</details>

**3.** 같은 투영·같은 알고리즘인데 동료가 뽑은 순위와 뒷자리가 다릅니다. 무엇부터 확인하나요?

<details><summary>정답 보기</summary>

**반복 횟수(`maxIterations`)와 수렴 여부**입니다. 기본값 20회는 수렴을 보장하지 않으므로, 점수가 붙어 있는 뒷자리는 설정에 따라 순서가 바뀝니다. `.stats` 의 `ranIterations`·`didConverge` 로 두 실행을 맞춰 보세요.

</details>

---
# 2. 실행 모드: 결과를 어디로 내보낼 것인가

계산한 점수를 어디에 둘 것인가가 실행 모드입니다. 화면으로 흘려보낼지, 투영 안에만 남길지, 원본 데이터베이스에 저장할지가 갈립니다.

## 2-1. `write` 로 원본에 남기고 `mutate` 로 투영에만 남기기

지금까지 쓴 `.stream` 은 **결과를 화면으로 흘려보내는** 모드입니다. GDS 알고리즘은 대부분 네 가지 모드를 갖습니다.

| 모드 | 하는 일 | 언제 쓰나 |
|---|---|---|
| `.stream` | 노드마다 점수를 한 행씩 돌려준다 | 지금 확인만 하고 싶을 때 |
| `.stats` | 요약 통계만 돌려준다(점수는 안 준다) | 분포만 보고 싶을 때 |
| `.mutate` | **투영 안에** 새 속성으로 써 넣는다 | 여러 결과를 모아 뒀다가 한 번에 꺼낼 때 |
| `.write` | **원본 데이터베이스**의 노드 속성으로 저장한다 | 나중에 Cypher 로 조회할 때 |

이름만 바꾸면 됩니다. 인자 모양은 같습니다.

```cypher
CALL gds.pageRank.write('투영이름', {writeProperty: '속성이름'})
YIELD nodePropertiesWritten
```

In [ ]:
# 계산한 PageRank 점수를 원본 노드의 pagerank 속성으로 저장한다
# 투영에 담긴 노드 전부에 값이 쓰인다. 관계가 없는 노드도 기본 점수를 받는다
written = run_cypher("""
    CALL gds.pageRank.write('drugGraph', {writeProperty: 'pagerank'})  // 계산 결과를 원본 노드의 속성으로 쓴다
    YIELD nodePropertiesWritten  // 여러 컬럼 중 개수만 골라 받는다. 점수 자체는 어느 컬럼에도 안 들어 있다
    RETURN nodePropertiesWritten
""")[0]["nodePropertiesWritten"]
print("속성이 쓰인 노드 수:", written)

In [ ]:
# MATCH 로 질병만 잡고, 점수가 안 쓰인 노드는 WHERE 로 걸렀다
rows = run_cypher("""
    MATCH (d:Disease)  // GDS 없이 평범한 Cypher 로 조회된다. 값이 원본에 남았기 때문이다
    WHERE d.pagerank IS NOT NULL  // 이 투영은 질병을 전부 담았으니 지금은 걸리는 행이 없다. 다른 투영으로 write 했다면 안 담긴 질병에 값이 없으니 습관으로 둔다
    RETURN d.name AS name, d.pagerank AS score
    ORDER BY score DESC, name
    LIMIT 5
""")
for row in rows:
    print(f"  {row['name']:26} {row['score']:.3f}")

> **`write` 의 값어치**: 점수를 원본에 남겨 두면 다른 조건과 **함께** 물을 수 있습니다. "PageRank 상위이면서 치료제가 5종 이상인 질병" 같은 질문은 Cypher 한 줄로 끝납니다. `stream` 만 쓰면 매번 다시 계산해야 합니다.

### 원본을 건드리지 않는 저장: `mutate`

표에서 본 `mutate` 를 직접 돌려 봅니다. `write` 와 문법이 거의 같고 **어디에 쓰느냐만** 다릅니다. `mutate` 는 계산한 값을 **투영 안에만** 넣습니다. 다음 알고리즘의 입력으로만 쓸 값이라면 원본을 더럽힐 이유가 없죠.

In [ ]:
# writeProperty 자리에 mutateProperty 를 쓴다. 이름만 다르고 모양은 write 와 같다
mutated = run_cypher("""
    CALL gds.pageRank.mutate('drugGraph', {mutateProperty: 'pr_in_memory'})  // write 와 모양이 같고 값이 가는 곳만 다르다. 투영 안에만 남는다
    YIELD nodePropertiesWritten
    RETURN nodePropertiesWritten
""")[0]["nodePropertiesWritten"]
print("투영 안에 값이 쓰인 노드 수:", mutated)

In [ ]:
# 투영 안의 값은 GDS 를 거쳐야 읽힌다. 이 결과가 나오면 투영에는 확실히 들어간 것이다
rows = run_cypher("""
    CALL gds.graph.nodeProperty.stream('drugGraph', 'pr_in_memory')  // 투영 안의 값은 이 프로시저를 거쳐야 읽힌다
    YIELD nodeId, propertyValue
    RETURN gds.util.asNode(nodeId).name AS name, propertyValue AS score
    ORDER BY score DESC, name
    LIMIT 3
""")
for row in rows:
    print(f"  {row['name']:26} {row['score']:.3f}")

In [ ]:
# 반대로 평범한 Cypher 로는 안 보인다. mutate 는 원본에 아무것도 남기지 않는다
# 없는 속성을 직접 짚으면 드라이버가 경고를 띄우니, keys(n) 로 이름을 뒤진다
left = run_cypher("MATCH (n) WHERE 'pr_in_memory' IN keys(n) RETURN count(n) AS c")[0]["c"]
print("원본에서 pr_in_memory 를 가진 노드:", left)

> **이것이 `write` 와 `mutate` 의 전부입니다.** 같은 점수를 계산했는데 `write` 는 원본에서 조회되고 `mutate` 는 0건입니다. 투영을 내리면(`gds.graph.drop`) `mutate` 로 넣은 값은 함께 사라집니다.

### 그러면 `mutate` 는 언제 쓰나

값이 투영과 함께 사라진다면 왜 쓸까요. **여러 알고리즘의 결과를 한자리에 모으기 위해서**입니다. `stream` 은 부를 때마다 결과가 따로 오지만, `mutate` 로 쌓아 두면 **한 번의 조회로 나란히** 꺼낼 수 있습니다.

바꿀 것은 하나뿐입니다. `mutateProperty` 이름만 바꿔 여러 번 부르면 됩니다.

In [ ]:
# 알고리즘 셋을 같은 투영에 쌓는다. 이름만 다르고 호출 모양은 똑같다
# 매개 중심성은 5절에서 배운다. 여기서는 '이름만 바꿔 여러 번 부른다' 를 보이려고 미리 한 번 돌린다
for _algo, _prop in [("degree", "deg"), ("pageRank", "pr"), ("betweenness", "bc")]:
    run_cypher(f"CALL gds.{_algo}.mutate('drugGraph', {{mutateProperty: '{_prop}'}}) "
               "YIELD nodePropertiesWritten RETURN nodePropertiesWritten")
print("세 결과를 투영에 쌓았습니다")

In [ ]:
# 복수형 nodeProperties.stream 은 속성 여러 개를 한 번에 준다(앞의 단수형은 하나만 준다)
# 한 노드의 세 값이 세 행으로 오므로, 이름으로 묶어 한 줄로 만든다
rows = run_cypher("""
    CALL gds.graph.nodeProperties.stream('drugGraph', ['deg', 'pr', 'bc'])
    YIELD nodeId, nodeProperty, propertyValue
    WITH gds.util.asNode(nodeId).name AS name, nodeProperty, propertyValue
    WITH name, collect([nodeProperty, propertyValue]) AS vals
    WITH name, apoc.map.fromPairs(vals) AS m   // [[키, 값], ...] 목록을 {키: 값} 맵으로 바꾼다
    RETURN name, m.deg AS 차수, m.pr AS PageRank, m.bc AS 매개
    ORDER BY PageRank DESC, name LIMIT 5
""")
for row in rows:
    print(f"  {row['name']:26} 차수 {int(row['차수']):>3}  "
          f"PageRank {row['PageRank']:6.3f}  매개 {row['매개']:>10,.0f}")

**세 잣대가 한 표에 나란히 섰습니다.** `stream` 을 세 번 불러 파이썬에서 합치는 것과 결과는 같지만, **계산은 투영 위에서 한 번씩만** 일어나고 조회도 한 번입니다. 결과를 투영에 모아 두었다가 한 번에 꺼내야 할 때 쓰는 방식입니다(6절은 이미 `stream` 으로 받아 둔 세 결과를 그대로 나란히 놓습니다).

여기까지 오면 남은 질문은 하나입니다. **이 값들을 원본에 남기려면?** 알고리즘마다 `.write` 를 다시 부를 필요가 없습니다. 투영에 쌓아 둔 것을 **한 번에** 넘기는 프로시저가 따로 있습니다.

```cypher
CALL gds.graph.nodeProperties.write('투영이름', ['deg', 'pr', 'bc'])
YIELD propertiesWritten
```

> **실무의 흐름이 이 모양입니다.** 투영을 한 번 만들고 → 필요한 알고리즘을 `mutate` 로 여러 번 돌리고 → 마지막에 `nodeProperties.write` 로 한 번에 저장하고 → 투영을 내립니다. 알고리즘마다 `.write` 를 부르면 원본에 쓰는 일이 그만큼 여러 번 일어납니다.

### 🖐️ 함께 따라하기: 전철 그래프의 점수를 속성으로 남기기

`subwayGraph` 의 PageRank 를 **`station_rank`** 라는 속성으로 저장하고, 저장된 속성으로 `Station` 노드 상위 3개를 조회해 출력하세요.

**확인 기준**: 속성이 쓰인 노드 수는 659개, 조회 1위는 `김포공항` 입니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) gds.pageRank.write('subwayGraph', {writeProperty: 'station_rank'}) 를 실행하고
#    YIELD nodePropertiesWritten 값을 출력한다
# 2) MATCH (s:Station) 으로 station_rank 상위 3개를 조회해 출력한다

### ✅ 바로 확인 퀴즈

**1.** `.write` 와 `.mutate` 의 차이는 무엇인가요?

<details><summary>정답 보기</summary>

`.write` 는 **원본 데이터베이스**의 노드 속성으로 저장하고, `.mutate` 는 **투영 안에만** 써 넣습니다. 다음 알고리즘의 입력으로만 쓸 값이면 `.mutate`, 나중에 Cypher 로 조회할 값이면 `.write` 입니다.

</details>

**2.** 점수만 화면으로 확인하고 끝낼 때 알맞은 모드는?

<details><summary>정답 보기</summary>

`.stream` 입니다. 아무것도 저장하지 않고 결과 행만 돌려줍니다.

</details>

---
# 3. 개인화 PageRank: 출발점을 정하고 다시 계산한다

여기서는 `sourceNodes` 로 출발점을 정해 순위를 다시 뽑습니다. 그리고 같은 질문을 두 투영에서 던져 보고, 답이 갈리는 것으로 **투영이 곧 질문의 범위**임을 확인합니다.

## 3-1. 출발점을 정해 관점을 바꾸기

### 왜 필요할까요?
지금까지의 PageRank 는 **모두에게 같은 답**을 줍니다. 그런데 실무 질문은 대개 이런 모양입니다.

> "**Sildenafil 와 가까운 것**은 무엇인가?"

전체 순위는 이 질문에 답하지 못합니다. **출발점을 정하고** 그 자리에서 다시 계산해야 합니다. 그것이 **개인화 PageRank(Personalized PageRank)** 입니다.

### 식에서 바뀌는 것은 상수항 하나입니다
맨 앞에서 본 정의를 다시 놓고 봅니다. 앞항 $1 - d$ 는 **모든 노드가 똑같이 받는 몫**이었습니다.

$$ PR(v) = (1 - d) \; + \; d \sum_{u \,\in\, In(v)} \frac{PR(u)}{L(u)} $$

개인화 PageRank 는 이 앞항에 **스위치**를 답니다.

$$ PPR(v) = (1 - d)\, s(v) \; + \; d \sum_{u \,\in\, In(v)} \frac{PPR(u)}{L(u)} $$

$s(v)$ 는 **출발점이면 1, 그 밖이면 0** 입니다. `sourceNodes` 에 넘긴 노드가 $s(v) = 1$ 이 되는 자리입니다. 바뀐 것은 이것뿐이고 뒤의 합산 항은 글자 하나 다르지 않습니다. 그런데 이 스위치 하나가 세 가지를 바꿉니다.

| | 일반 PageRank | 개인화 PageRank |
|---|---|---|
| 새 점수를 받는 곳 | **모든 노드**가 매 반복 $1-d$ 씩 | **출발점만** 매 반복 $1-d$ 씩 |
| 가장 낮은 점수 | $1-d =$ `0.15`. 이웃이 없어도 이만큼은 받는다 | `0`. 출발점에서 갈 수 없으면 아무것도 못 받는다 |
| 점수 총합 | **노드 수**(2,012)에 가깝게 모인다(정확히 같아지는 조건은 아래 참고) | **출발점 개수**에 맞춰진다(하나면 1) |

1. **출발점이 1위로 나오는 것이 보통입니다.** 출발점만 매 반복 $1-d$ 를 새로 받고, 다른 노드는 흘러온 것만 갖습니다.
2. **점수가 `0` 인 노드가 생깁니다.** 일반 PageRank 는 바닥이 `0.15` 라 `0` 이 나올 수가 없었습니다. "관점을 정한다"는 말의 뜻이 이것입니다.

   다만 **`0` 이 곧 "못 간다"는 뜻은 아닙니다.** 이 투영에서 `Sildenafil` 로부터 한 걸음도 닿지 않는 노드는 222개인데, 기본 설정으로 돌리면 0점이 **236개** 나옵니다. 14개가 더 많죠. 바로 위에서 본 그 `tolerance` 가 아주 작은 몫의 전파를 중간에 끊어서, 닿기는 하는데 **멀어서** 0 이 된 노드가 섞이기 때문입니다. 둘을 가르려면 `tolerance: 0.0` 으로 다시 돌리세요. 그러면 0점이 정확히 222개, 곧 **정말 못 가는 노드만** 남습니다.
3. **두 점수를 견주지 마세요.** 총합이 다르니 자릿수부터 다릅니다. 잠시 뒤 개인화 1위 점수는 `0.21` 쯤으로 나오는데, 앞에서 본 일반 PageRank 1위 `8.08` 보다 작다고 해서 덜 중요한 것이 아닙니다. **각자의 순위표 안에서만** 읽습니다.

> **총합이 왜 그 값인가**: 식의 양변을 모든 노드에 대해 더하면 나옵니다. 앞항의 합은 일반이 $(1-d)N$, 개인화가 $(1-d)|S|$ 입니다($|S|$ 는 출발점 개수). 뒷항은 노드 $u$ 가 자기 점수를 $L(u)$ 갈래로 나눠 보낸 것을 **다시 다 모으는** 것이라 $d\,T$ 가 됩니다. 정리하면 $T = N$ · $T = |S|$ 입니다. **$d$ 가 약분돼 사라지므로 감쇠 계수를 얼마로 두든 총합은 같습니다.**

> **단, 점수가 다음 회로 흘러갈 수 있어야** 이 등식이 성립합니다. 선이 하나도 없는 노드는 받은 $1-d$ 를 넘길 데가 없어 그만큼이 순환에서 빠집니다. 이 조각에는 그런 노드가 104개라 일반 PageRank 총합이 이론값 2,012 가 아니라 **1,923.6** 에서 수렴합니다(반복을 2,000회로 올려도 같습니다). 선이 있는 노드만 담아 다시 투영하면 총합이 노드 수와 일치합니다. 다만 여기서도 소수점까지 딱 맞추려면 `tolerance: 0` 을 함께 주어야 합니다. 개인화 쪽은 고립 노드가 점수를 아예 못 받으므로(0점) 새는 것이 없어 총합이 출발점 개수가 됩니다.

> 기본 설정(`maxIterations: 20`)에서는 개인화 총합이 `0.96` 쯤입니다. 아직 수렴 전이기 때문입니다. 다만 **반복만 늘려서는 `1.0000` 에 닿지 않습니다.** 기본 `tolerance`(1e-7)가 46회에서 "충분히 안 움직인다"고 판정해 멈추기 때문입니다(`0.9979`). `tolerance: 0` 까지 함께 주어야 정확히 `1.0000` 이 됩니다.

```cypher
MATCH (s:Compound {name: '약이름'})
WITH collect(s) AS src
CALL gds.pageRank.stream('투영이름', {sourceNodes: src}) YIELD nodeId, score
```

<img src="images/개인화_pagerank.png" width="860">

In [ ]:
# drugGraph(약물·질병·약효분류 조각)에서 Sildenafil 를 출발점으로 정해 다시 계산한다
# 모두에게 같은 답을 주던 PageRank 가, 이 셀부터 '이 약에서 본 순위' 로 바뀐다
ppr_drug = run_cypher("""
    MATCH (s:Compound {name: $name})
    WITH collect(s) AS src  // sourceNodes 는 노드 하나가 아니라 노드 목록을 받는다
    CALL gds.pageRank.stream('drugGraph', {sourceNodes: src})  // sourceNodes 가 붙는 순간 '모두의 순위' 가 '이 노드에서 본 순위' 로 바뀐다
    YIELD nodeId, score
    RETURN gds.util.asNode(nodeId).name AS name,
           labels(gds.util.asNode(nodeId))[0] AS kind,
           score
    ORDER BY score DESC, name
    LIMIT 6
""", name="Sildenafil")
# 1위는 출발점 자신이다. 점수가 거기서 나오기 때문이다
for rank, row in enumerate(ppr_drug, 1):
    print(f"{rank}. {row['name']:26} {row['kind']:12} {row['score']:.4f}")

In [ ]:
# 위 표에서 말한 0점 개수를 직접 센다. 기본 설정과 tolerance 0 이 왜 다른지 눈으로 확인하는 셀이다
for label, extra in [('기본 설정', ''), ('tolerance: 0', ', tolerance: 0.0')]:
    zeros = run_cypher(f'''
        MATCH (s:Compound {{ name: $name }})
        WITH collect(s) AS src
        CALL gds.pageRank.stream('drugGraph', {{ sourceNodes: src{extra} }})
        YIELD score
        RETURN count(CASE WHEN score = 0 THEN 1 END) AS zeros''',
        name='Sildenafil')[0]['zeros']
    print(f'{label:12} 0점 노드 {zeros}개')
# 두 수의 차이가 '멀어서 0' 이 된 노드다. tolerance 0 쪽이 정말 못 가는 노드의 수와 같다

닮은 약 두 개가 바로 올라왔습니다. **Sildenafil 는 전체 PageRank 로는 1,000위권**인데, 출발점을 그 노드로 잡으니 그 주변이 순위표를 차지합니다. 관점이 바뀐 것입니다.

### 그런데 이 답에는 빠진 것이 있습니다

"닮은 약"까지는 나왔는데 **왜 닮았는지**는 안 나왔습니다. 약이 실제로 결합하는 **표적**이 있을 텐데 그 이야기가 없죠. `drugGraph` 에 **`BINDS` 관계를 담지 않았기 때문**입니다. 이 약만 해도 `BINDS` 가 17건 있는데, 투영에 없으니 계산에 끼어들 수가 없습니다.

> **투영이 곧 질문의 범위입니다.** 담지 않은 관계는 존재하지 않는 것과 같습니다. 답이 틀린 게 아니라 **답할 수 있는 질문의 종류가 달라지는** 것입니다.

> **`Gene` 노드를 읽는 법**: 약이 실제로 붙는 것은 유전자(DNA)가 아니라 **그 유전자가 만드는 단백질**입니다. 이 데이터는 유전자와 그 산물인 단백질을 **`Gene` 노드 하나로** 다루고 이름도 유전자 이름을 씁니다. 그래서 `PDE5A` 는 유전자 이름이면서 그 유전자가 만드는 단백질의 이름이기도 합니다. 아래에서 "표적"이라고 부르는 것이 이 단백질입니다.

그래서 이 질문에는 **전체를 담은 투영**이 필요합니다.

In [ ]:
# 레이블 5종과 관계 12종을 전부 담은 투영. 유전자·증상까지 들어온다
# 무방향은 관계 타입마다 따로 적어야 한다. 하나라도 빠뜨리면 그 타입만 방향이 남는다
res = run_cypher("""
    CALL gds.graph.project('fullGraph',
        ['Compound', 'Disease', 'Gene', 'Symptom', 'PharmacologicClass'],
        {TREATS:           {orientation: 'UNDIRECTED'},
         PALLIATES:        {orientation: 'UNDIRECTED'},
         BINDS:            {orientation: 'UNDIRECTED'},
         UPREGULATES_CG:   {orientation: 'UNDIRECTED'},
         DOWNREGULATES_CG: {orientation: 'UNDIRECTED'},
         ASSOCIATES:       {orientation: 'UNDIRECTED'},
         UPREGULATES_DG:   {orientation: 'UNDIRECTED'},
         DOWNREGULATES_DG: {orientation: 'UNDIRECTED'},
         RESEMBLES_DD:     {orientation: 'UNDIRECTED'},
         RESEMBLES_CC:     {orientation: 'UNDIRECTED'},
         PRESENTS:         {orientation: 'UNDIRECTED'},
         INCLUDES:         {orientation: 'UNDIRECTED'}})
    YIELD nodeCount, relationshipCount  // 만들어진 투영의 크기를 바로 돌려받아 검산에 쓴다
    RETURN nodeCount, relationshipCount
""")[0]
# 검산: 무방향이니 원본 관계 수의 2배여야 한다(지난 시간의 습관)
# 다만 이 DB 에는 전철 그래프도 함께 있어서 관계를 전부 세면 안 된다. NEXT_TO·ON_LINE 이
# 섞여 들어와 2배가 어긋난다. 투영에 담은 12종만 세는 것이 옳다(담은 것만 센다)
# 노드는 걸러 낼 필요가 없다. 위에서 레이블 5종을 이름으로 적어 담았으니 역·노선은 애초에 없다
raw = run_cypher("""
    MATCH ()-[r]->()
    WHERE type(r) IN ['TREATS', 'PALLIATES', 'BINDS', 'UPREGULATES_CG',
                      'DOWNREGULATES_CG', 'ASSOCIATES', 'UPREGULATES_DG', 'DOWNREGULATES_DG',
                      'RESEMBLES_DD', 'RESEMBLES_CC', 'PRESENTS', 'INCLUDES']  // fullGraph 에 담은 12종만 센다
    RETURN count(r) AS cnt
""")[0]["cnt"]
print(res, "/ 원본", raw, "의 2배인가?", res["relationshipCount"] == raw * 2)


이 투영이 담은 것을 한 장으로 보면 이렇습니다. 앞의 `drugGraph` 와 견주어 보세요. 노드 세 종류가 다섯 종류로, 관계 다섯 종류가 열두 종류로 늘었습니다.

<img src="images/fullGraph_구성.png" width="900">

In [ ]:
# 출발점은 그대로 Sildenafil 다. 투영만 fullGraph 로 바꿔 같은 질문을 다시 던진다
# 유전자·증상까지 담긴 그래프라, 약끼리만 보던 앞 셀과 답이 달라진다
ppr_full = run_cypher("""
    MATCH (s:Compound {name: $name})
    WITH collect(s) AS src
    CALL gds.pageRank.stream('fullGraph', {sourceNodes: src})  // 투영 이름만 바꿨다. 나머지는 앞 셀과 같다
    YIELD nodeId, score
    RETURN gds.util.asNode(nodeId).name AS name,
           labels(gds.util.asNode(nodeId))[0] AS kind,
           score
    ORDER BY score DESC, name
    LIMIT 8
""", name="Sildenafil")
# 담은 것이 많아진 만큼 점수가 잘게 나뉜다. 자릿수를 하나 늘려 찍는다
for rank, row in enumerate(ppr_full, 1):
    print(f"{rank}. {row['name']:26} {row['kind']:12} {row['score']:.5f}")

이제 **유전자가 함께 나옵니다.**

- `Vardenafil` 는 같은 계열의 약입니다.
- `PDE5A` 는 이 약이 실제로 결합하는 **표적**입니다. 유전자 이름으로 적혀 있지만 약이 붙는 것은 그 유전자가 만드는 단백질입니다. 약리학 교과서에 나오는 표적이 **그래프 계산만으로** 상위에 올라온 것입니다.
- `CYP3A4`·`CYP2D6` 처럼 약을 대사하는 효소도 보입니다. 여러 약이 함께 붙는 흔한 효소죠.

### 이 순위는 "가깝다"가 아닙니다
이 점수를 **거리**로 읽으면 틀립니다. 앞의 식이 세는 것은 몇 걸음 떨어졌는지가 아니라, **출발점에서 아무 선이나 따라 걸어 다닐 때 그 노드를 얼마나 자주 지나치는가**입니다. 그래서 세 가지가 함께 점수를 정합니다.

| 무엇이 점수를 올리나 | 식의 어느 부분 |
|---|---|
| **받을 통로가 많다**: 그 노드로 이르는 경로가 여럿이다 | 합산 항의 **개수**가 는다 |
| **통로마다 오는 몫이 크다**: 보내 주는 이웃의 선이 적다 | 이웃이 $L(u)$ 로 나눠 넘긴다 |
| **출발점에서 가깝다** | 한 걸음마다 $d$ 배씩 깎인다 |

> **헷갈리는 자리**: "선이 많으면 몫이 깎인다"는 **보내는 쪽** 이야기입니다. $L(u)$ 는 **이웃 $u$** 의 선 개수이고, $PR(v)$ 식 어디에도 **$L(v)$ 는 없습니다.** 내 선이 많은 것은 **받을 통로가 많다**는 뜻(표 첫째 줄)이라 오히려 유리합니다. 실제로 이 순위표 2위 `CYP3A4` 는 선이 **521개**입니다.

순위표 7위 **`Dipyridamole`** 이 세 가지가 부딪히는 자리입니다. 이 약은 Sildenafil 의 직접 이웃이 **아닌데도**(2홉) 1홉 이웃 28개 중 **23개보다 위**에 있습니다. `PDE5A`·`PDE6C`·`PDE6G`·`PDE6H`·`ABCC4`·`ABCC5` **여섯 통로**로 이어져 있어서입니다.

좋은 대조가 하나 있습니다. **`PDE6C`** 는 Sildenafil 의 **직접 이웃**인데 점수가 `0.00504` 로 Dipyridamole(`0.00717`)보다 **아래**입니다. 이 유전자는 선이 둘뿐이라(Sildenafil 과 Dipyridamole) **받을 통로가 둘**입니다. 게다가 출발점에서 오는 몫은 Sildenafil 의 점수를 **선 28개로 나눈 것**이라 한 통로가 크지 않습니다.

Dipyridamole 은 **여섯 통로**로 받습니다. 그중 `PDE6C`·`PDE6H`·`PDE6G` 는 선이 각각 **2·3·4개**뿐이라, 받은 몫의 큰 덩어리를 그대로 Dipyridamole 에 넘깁니다. **통로가 셋 더 많고 통로마다 오는 몫도 큰 것**이, 한 걸음 가까운 것을 이깁니다.

거리로 줄을 세우는 것은 최단 경로가 하는 일이고, 34일차에서 따로 배웁니다.

> **읽을 때 주의**: 이 순위가 말하는 것은 **이 그래프 안에서의 관계 구조**일 뿐입니다. **"이 약이 저 병에 듣는다"는 뜻이 아닙니다.** 데이터는 문헌에 보고된 연관을 모은 것이고, 효능은 임상시험이 판단합니다. 그래프 분석은 "살펴볼 후보를 좁히는" 도구입니다.

### 두 투영, 두 질문

<img src="images/투영이_질문의_범위.png" width="860">

| 질문 | 알맞은 투영 | 이유 |
|---|---|---|
| 이 그래프에서 무엇이 중심인가 | **`drugGraph`** | 전체로 재면 연결이 많은 질병이 상위를 다 차지한다 |
| 이 약과 이어진 것은 무엇인가 | **`fullGraph`** | 약과 표적을 잇는 `BINDS` 가 있어야 답이 나온다 |

실제로 전체 투영에서 그냥 PageRank 를 돌리면 어떻게 되는지 확인해 봅니다.

In [ ]:
# 전체 투영에서 개인화 없이 PageRank 를 돌려 본다. 무엇이 상위를 차지하나
rows = run_cypher("""
    CALL gds.pageRank.stream('fullGraph')  // sourceNodes 를 빼면 다시 '모두에게 같은 답' 이 된다
    YIELD nodeId, score
    RETURN labels(gds.util.asNode(nodeId))[0] AS kind, gds.util.asNode(nodeId).name AS name
    ORDER BY score DESC, name
    LIMIT 10
""")
# 노드 종류만 세어 본다. 열 자리를 무엇이 나눠 갖는지가 요점이다
kinds = [row["kind"] for row in rows]
print("상위 10 의 노드 종류:", {k: kinds.count(k) for k in set(kinds)})

In [ ]:
print("1위:", rows[0]["name"])

열 자리가 **전부 질병**입니다. 유전자 13,113개가 질병 몇 개에 방사형으로 달라붙은 구조라, 전체로 재면 질병이 이길 수밖에 없습니다. "어떤 약이 중요한가"를 묻고 싶다면 이 투영은 쓸 수 없습니다.

**그래서 투영이 두 개입니다.** 질문마다 담을 것이 다릅니다.

### 🖐️ 함께 따라하기: 한 역 관점에서 본 전철 그래프

시연은 **약물 기준**이었습니다. 따라하기는 `subwayGraph` 에서 **역 기준**으로 합니다.

`강남` 역을 출발점으로 개인화 PageRank 를 돌려 상위 5개를 출력하세요. 출발점 자신은 1위로 나오니 **그다음부터** 읽으면 됩니다.

**확인 기준**: 1위는 출발점 자신, 2위 `양재`, 3위 `교대` 입니다. 1절 따라하기에서 전체 1위였던 `김포공항` 이 이 순위표에 나오는지도 함께 보세요.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) MATCH (s:Station {name: '강남'}) WITH collect(s) AS src 로 출발 노드를 만든다
# 2) gds.pageRank.stream('subwayGraph', {sourceNodes: src}) 를 실행한다
# 3) 이름과 점수를 내림차순 5개 출력한다

2위 `양재` 와 3위 `교대` 는 `강남` 에서 신분당선·2호선으로 **바로 이어진 이웃 역**입니다.

정작 눈여겨볼 것은 순위표에 **없는 이름**입니다. 전체 순위 1위였던 `김포공항` 이 여기에는 아예 나오지 않습니다. 그래프에서 가장 중심인 역인데도요. 출발점을 정하는 순간 순위표는 **그 주변의 것**이 됩니다. 시연에서 약으로 본 것과 같은 일이 도메인만 바뀌어 그대로 일어납니다.

### ✅ 바로 확인 퀴즈

**1.** 개인화 PageRank 에서 출발 노드가 대체로 1위로 나오는 이유는?

<details><summary>정답 보기</summary>

식의 앞항 $(1-d)\,s(v)$ 를 **출발점만** 받기 때문입니다. 출발점은 매 반복 새 점수를 받고, 다른 노드는 이웃에게서 흘러온 것만 갖습니다. 다만 항상은 아닙니다. 출발점의 선이 아주 적고 그 이웃에 선이 몰려 있으면 이웃이 더 높게 나올 수 있습니다.

</details>

**2.** 같은 약을 같은 방식으로 물었는데 `drugGraph` 와 `fullGraph` 의 답이 다릅니다. 어느 쪽이 틀린 건가요?

<details><summary>정답 보기</summary>

**어느 쪽도 틀리지 않았습니다.** 투영에 담긴 관계가 다르니 답할 수 있는 질문이 다를 뿐입니다. `BINDS` 를 담지 않은 투영은 "어느 표적에 붙나"에 답할 수 없습니다. 무엇을 담을지 고르는 것이 곧 질문을 정하는 일입니다.

</details>

---
# 4. 계산 범위를 좁히는 두 가지 필터

투영을 새로 만들지 않고도 **이번 계산에만** 범위를 좁힐 수 있습니다. 다만 좁히는 방식이 결과를 거르는 것이 아니라서, 답 자체가 바뀝니다.

## 4-1. `nodeLabels` 로 노드 종류 줄이기(그리고 `relationshipTypes` 와 함께 쓰기)

알고리즘 설정에 두 가지를 넣으면 됩니다.

| 설정 | 뜻 |
|---|---|
| `nodeLabels: ['Compound']` | 이 레이블의 노드만 두고 계산한다 |
| `relationshipTypes: ['BINDS']` | 이 타입의 관계만 두고 계산한다 |

> **중요**: 이것은 **결과를 걸러내는 것이 아니라 그래프 자체를 줄이는 것**입니다. `Compound` 만 남기면 약과 유전자를 잇던 `BINDS` 도 함께 사라집니다(양끝이 다 있어야 관계가 남는다는 그 규칙입니다). 그래서 **답이 달라집니다.**

In [ ]:
# Sildenafil 를 출발점으로 둔 개인화 PageRank 를, Compound 노드만 남기고 다시 돌린다
# 결과를 걸러내는 것이 아니라 그래프 자체를 줄이는 것이다. 그래서 답 자체가 달라진다
only_drug = run_cypher("""
    MATCH (s:Compound {name: $name})
    WITH collect(s) AS src
    CALL gds.pageRank.stream('fullGraph', {sourceNodes: src, nodeLabels: ['Compound']})  // 이 레이블만 남기고 계산한다. 나머지 노드와 그 관계는 사라진다
    YIELD nodeId, score
    RETURN gds.util.asNode(nodeId).name AS name, score
    ORDER BY score DESC, name
    LIMIT 5
""", name="Sildenafil")
for rank, row in enumerate(only_drug, 1):
    print(f"{rank}. {row['name']:20} {row['score']:.5f}")

0 점인 약이 곧바로 따라 나옵니다. **약물끼리 잇는 관계는 `RESEMBLES_CC` 하나뿐**이라, 그 선으로 닿지 않는 약은 전부 0 이 됩니다. 답이 훨씬 뾰족해졌지만 그만큼 좁아졌습니다.

관계 타입으로 좁히면 또 다른 답이 나옵니다.

In [ ]:
# 역시 Sildenafil 출발점의 개인화 PageRank 다. 이번엔 BINDS 관계만 남기고 돌린다
# BINDS = 약이 그 유전자가 만드는 단백질에 결합한다. 즉 '이 약이 붙는 표적' 만 남긴 그래프다
# relationshipTypes 는 노드는 그대로 두고 관계만 고른다. 남은 관계로 닿지 않는 노드는 0 점이 된다
only_binds = run_cypher("""
    MATCH (s:Compound {name: $name})
    WITH collect(s) AS src
    CALL gds.pageRank.stream('fullGraph', {sourceNodes: src, relationshipTypes: ['BINDS']})  // 노드는 그대로 두고 BINDS 관계만 남긴다
    YIELD nodeId, score
    RETURN gds.util.asNode(nodeId).name AS name,
           labels(gds.util.asNode(nodeId))[0] AS kind, score
    ORDER BY score DESC, name
    LIMIT 6
""", name="Sildenafil")
for rank, row in enumerate(only_binds, 1):
    print(f"{rank}. {row['name']:14} {row['kind']:10} {row['score']:.5f}")

같은 출발점, 같은 알고리즘인데 **세 번 다 다른 답**이 나왔습니다.

| 설정 | 답의 성격 |
|---|---|
| 필터 없음 | 닮은 약 + 표적 단백질 + 대사 효소가 섞여 나온다 |
| `nodeLabels: ['Compound']` | 화학적으로 닮은 약만 남는다 |
| `relationshipTypes: ['BINDS']` | 같은 표적에 붙는 약과 그 표적이 나온다 |

**셋 다 맞는 답입니다.** 다만 서로 다른 질문에 답하고 있습니다. 무엇을 남길지 정하는 순간 질문도 정해집니다.

### 🖐️ 함께 따라하기: '같은 약으로 다루는 병' 찾기

`fullGraph` 에서 `asthma`(천식)를 출발점으로 개인화 PageRank 를 돌리되, **`relationshipTypes: ['TREATS', 'PALLIATES']`** 로 약과 병을 잇는 관계만 남기세요. 상위 5개를 출력합니다.

**확인 기준**: 상위 5개에 `asthma` 자신과 천식 치료제, 그리고 그 약을 함께 쓰는 다른 병(`allergic rhinitis`, `chronic obstructive pulmonary disease` 같은 영문 이름)이 섞여 나옵니다. 노드 종류가 `Disease` 와 `Compound` 두 가지로 나오면 맞습니다. 천식과 **치료제를 나눠 쓰는 병들**과 그 약이 함께 올라옵니다. 관계를 좁히니 "같은 약으로 다루는 병"이라는 질문이 된 것입니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) MATCH (s:Disease {name: 'asthma'}) WITH collect(s) AS src
# 2) gds.pageRank.stream('fullGraph', {sourceNodes: src, relationshipTypes: ['TREATS','PALLIATES']})
# 3) 이름·노드 종류·점수를 내림차순 5개 출력한다

### ✅ 바로 확인 퀴즈

**1.** `nodeLabels: ['Compound']` 를 주면 결과에서 유전자가 빠지는 것 말고 또 무엇이 달라지나요?

<details><summary>정답 보기</summary>

**남은 노드들의 점수 자체가 달라집니다.** 유전자를 거쳐 가던 길이 사라지므로 계산이 완전히 다시 이뤄집니다. 결과를 걸러내는 것과는 다릅니다.

</details>

**2.** 투영을 새로 만드는 것과 `nodeLabels` 로 좁히는 것 중 어느 쪽이 편할까요?

<details><summary>정답 보기</summary>

**한 번만 좁혀 볼 때는 필터**가 편합니다. 투영을 만들고 지우는 비용이 없습니다. 같은 범위로 여러 알고리즘을 반복해 돌릴 거라면 그 범위로 투영을 따로 만드는 편이 낫습니다.

</details>

---
# 5. 매개 중심성: 길목에 선 노드

여기서는 `gds.betweenness.stream` 으로 세 번째 순위를 뽑습니다. 앞의 두 순위에 없던 이름이 왜 위로 올라오는지, 그 노드의 이웃을 직접 뜯어보며 확인합니다.

## 5-1. 길목에 선 노드 찾기

### 왜 필요할까요?
차수와 PageRank 는 둘 다 "연결이 몰린 곳"을 찾습니다. 그런데 다른 종류의 중요함이 있습니다. **혼자서 두 무리를 잇는 다리**입니다. 선이 몇 개 없어도, 그것이 없으면 길이 끊깁니다.

**매개 중심성(betweenness)** 은 "모든 노드 쌍의 최단 경로 중 몇 개가 나를 지나가는가"를 셉니다. 한 쌍에 가장 짧은 길이 여럿이면 그 몫을 나눠 셉니다. 다리 위에 선 노드일수록 높습니다.

```cypher
CALL gds.betweenness.stream('투영이름') YIELD nodeId, score
```

> **방향을 지운 투영이 필요합니다.** 최단 경로를 세는 알고리즘이라 화살표를 거슬러 못 가면 경로가 거의 안 생깁니다. `drugGraph` 는 무방향으로 만들어 뒀으니 그대로 쓰면 됩니다.

In [ ]:
# 최단 경로가 가장 많이 지나가는 노드 10개
# 방향이 남은 투영에서는 경로가 거의 안 생긴다. drugGraph 를 무방향으로 만들어 둔 이유다
# 모든 노드 쌍의 경로를 세므로 앞의 알고리즘보다 오래 걸린다. 이 조각은 1초 안에 끝나지만 큰 그래프에서는 분 단위가 된다
bc_rows = run_cypher("""
    CALL gds.betweenness.stream('drugGraph')  // 모든 노드 쌍의 최단 경로를 세므로 앞의 알고리즘들보다 오래 걸린다
    YIELD nodeId, score
    RETURN gds.util.asNode(nodeId).name AS name,
           labels(gds.util.asNode(nodeId))[0] AS kind, score
    ORDER BY score DESC, name
    LIMIT 10
""")
for rank, row in enumerate(bc_rows, 1):
    print(f"{rank:2}. {row['name']:26} {row['kind']:20} {row['score']:>12,.0f}")

1위는 여전히 `hypertension` 입니다. 하지만 **2위 `Phenacemide`** 는 앞의 두 순위에서 본 적이 없는 이름입니다. 차수로는 **162위**밖에 안 됩니다. 왜 길목일까요?

In [ ]:
# 매개 2위 노드가 어떤 약들과 닮았다고 되어 있는지 본다
rows = run_cypher("""
    MATCH (c:Compound {name: $name})-[:RESEMBLES_CC]-(other:Compound)  // 화학적으로 닮았다고 기록된 약만 본다. 길목 노드가 무엇과 무엇을 잇는지 보려는 것이다
    RETURN other.name AS neighbor
    ORDER BY neighbor
""", name="Phenacemide")
names = [row["neighbor"] for row in rows]
print("닮은 약 수:", len(names))

In [ ]:
print("예:", names[:4], "...", names[-3:])

### 이 알고리즘만 유독 느립니다

코드 주석에 "몇 초 걸린다"고만 적었는데, 왜 그런지 알아 두면 실무에서 크게 아낍니다. 매개 중심성은 **모든 노드 쌍의 최단 경로**를 셉니다. 노드가 늘면 쌍은 제곱으로 늘죠. 앞의 알고리즘들과 견주면 차이가 분명합니다.

| 투영 | PageRank | 매개 중심성 |
|---|---|---|
| `drugGraph`(노드 2,012) | `0.01`초 | `0.15`초 |
| `fullGraph`(노드 15,540) | `0.04`초 | **`16.15`초** |

노드가 **7.7배** 늘 때 PageRank 는 4배 느려지는데 매개 중심성은 **108배** 느려집니다. 3절에서 `drugGraph` 를 `fullGraph` 로 바꿔 같은 질문을 다시 던졌던 것을 기억하실 겁니다. 그 치환을 여기서 하면 0.15초가 16초가 됩니다. 실무 그래프에서는 분 단위가 됩니다.

**전부 세지 않고 표본만 세는 설정**이 있습니다.

```cypher
CALL gds.betweenness.stream('투영이름', {samplingSize: 1000})
```

출발점을 1,000개만 뽑아 경로를 셉니다. `fullGraph` 에서 `16.15`초가 **`1.05`초**로 줄어듭니다. 큰 그래프에서 먼저 감을 잡을 때 쓰는 표준 대응입니다.

> 표본을 뽑으므로 **실행할 때마다 값이 조금씩 달라집니다.** 순위를 인용할 때 왜 설정을 함께 적어야 하는지가 여기서도 나옵니다. 값을 고정하는 방법은 34일차에서 다룹니다.

> **가져갈 것**: 중심성은 호출 모양이 서로 똑같지만 **비용은 전혀 다릅니다.** 처음 보는 그래프에 매개 중심성을 그냥 돌리기 전에, 노드 수를 먼저 보고 표본부터 잡는 습관이 필요합니다.

닮은 약 26개에는 각성제(`Amphetamine`)·감미료(`Aspartame`)·항경련제(`Felbamate`)·항생제(`Benzylpenicillin`)가 한데 섞여 있습니다(위 출력은 이름 앞뒤만 잘라 보여 줍니다). 이 약의 화학 골격이 **서로 상관없는 여러 계열에 조금씩 걸쳐 있어서**, 계열과 계열 사이를 건너려면 이 노드를 지나가게 되는 것입니다. 선의 개수로는 보이지 않던 역할이죠.

**어느 계열과 어느 계열 사이의 다리인지** 한 쌍만 짚어 봅니다. `Benzylpenicillin` 은 **페니실린계 항생제**이고 `Methamphetamine` 은 **암페타민계 각성제**입니다. 약으로서 아무 상관이 없는 두 계열이죠.

| | 두 약을 잇는 가장 짧은 길 |
|---|---|
| `Phenacemide` 가 있을 때 | `Benzylpenicillin` → **`Phenacemide`** → `Methamphetamine` (2홉) |
| 이 노드를 빼면 | `Benzylpenicillin` → `Hetacillin` → `Ethotoin` → `Tranylcypromine` → `Phenylpropanolamine` → `Methamphetamine` (5홉) |

두 계열을 오가는 길이 `Phenacemide` 하나로 지나가고, 이 노드를 빼면 같은 두 약이 다섯 홉으로 멀어집니다. **이런 쌍이 계열 곳곳에 쌓인 결과**가 매개 중심성 2위입니다. 이 약 자신은 어느 계열의 대표도 아닌데 말이죠.

`Benzylpenicillin` 이 5위인 것도 같은 역할 때문입니다. 다만 이쪽은 **계열 안쪽의 관문**입니다. 닮은 약 23개가 거의 다 같은 베타락탐 계열이고, 계열 밖으로 나가는 이웃은 `Phenacemide` 하나뿐입니다. 그래서 페니실린계에서 바깥으로 나가는 길이 이 노드에 몰립니다.

> **실무에서 왜 보나**: 길목 노드는 **끊어지면 타격이 큰 자리**입니다. 통신망이면 장애 지점, 조직도면 정보가 몰리는 사람, 이 그래프에서는 서로 다른 약물 계열을 잇는 화학 골격입니다. 위 표처럼 **그 노드를 뺐을 때 길이 얼마나 길어지는가**로 타격의 크기를 가늠합니다.

### 🖐️ 함께 따라하기: 전철 그래프의 길목 찾기

`subwayGraph` 에서 매개 중심성 상위 5개를 뽑고, **앞에서 뽑은 차수 상위 5개와 나란히** 출력해 비교하세요(1절에서 만든 `top5` 함수를 다시 쓰면 됩니다). **1절 따라하기의 `top5` 를 먼저 완성해야 이 셀이 돕니다.** 비워 두고 왔다면 1절로 돌아가 그 셀부터 채우세요.

**확인 기준**: 매개 1위는 `김포공항` 으로 차수 1위와 같고, 2위는 `서울역` 입니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) 1절에서 만든 top5 함수를 'betweenness' 로 부른다
# 2) 같은 함수를 'degree' 로도 불러 두 목록을 만든다
# 3) pandas DataFrame 두 열로 만들어 display 한다

`top5` 의 `LIMIT` 를 10 으로 늘려 보면 매개 상위에 **`능곡`** 이 있습니다. 차수 상위 10 에는 없는 역인데요. 경의·중앙선과 서해선이 함께 지나는 자리라, **선 개수로는 안 보이던 역할이 매개로는 드러납니다.** 시연에서 본 `Phenacemide` 와 같은 자리입니다.

### ✅ 바로 확인 퀴즈

**1.** 매개 중심성이 높은데 차수는 낮은 노드는 어떤 자리에 있는 노드인가요?

<details><summary>정답 보기</summary>

**서로 다른 무리를 잇는 다리**입니다. 이웃 수는 적어도 그 노드를 거치지 않으면 두 무리 사이를 오갈 수 없습니다.

</details>

**2.** 매개 중심성을 방향이 남은 투영에서 계산하면 왜 곤란한가요?

<details><summary>정답 보기</summary>

최단 경로를 세는 알고리즘인데 화살표를 거스를 수 없으면 **경로가 거의 만들어지지 않습니다.** 실제로 같은 조각을 방향을 남긴 채 재면 점수 총합이 **19분의 1**로 줄고 0점 노드가 29.5%에서 46.3%로 늡니다. **에러는 안 나고 순위표도 그럴듯하게 나오는데 1위부터 다른 이름**이라, 검산하지 않으면 틀린 줄 모릅니다.

</details>

---
# 6. 세 잣대는 서로 다른 답을 낸다

오늘의 결론이 이 절의 표 하나에 담깁니다. 세 잣대가 정말 서로 다른 것을 재는지 숫자로 확인합니다.

## 6-1. 세 순위의 겹침을 세어 보기

오늘 계산한 세 순위를 한 표에 놓고 **몇 개나 겹치는지** 세어 봅니다. 셋 다 **같은 `drugGraph`** 에서 나온 값입니다. 약물·질병·약효분류 노드와 그 사이 관계 5종을 담은 그 조각이고, 데이터도 그대로입니다. **바뀐 것은 잣대 하나뿐**입니다.

In [ ]:
# 세 순위를 나란히 놓는다. 셋 다 drugGraph 하나에서 나온 값이다
# drugGraph = 약물(Compound)·질병(Disease)·약효분류(PharmacologicClass) 와 그 사이 관계 5종
# 같은 투영·같은 데이터에 잣대만 달랐다. 이름 뒤에 노드 종류를 붙여 무엇이 올라왔는지 함께 본다
three = pd.DataFrame({
    "차수": [f"{row['name']} ({row['kind']})" for row in deg_rows],
    "PageRank": [f"{row['name']} ({row['kind']})" for row in pr_rows],
    "매개 중심성": [f"{row['name']} ({row['kind']})" for row in bc_rows],
})
three.index = range(1, len(three) + 1)   # 인덱스를 1부터 매겨 순위로 읽히게 한다
display(three)

In [ ]:
# 쌍마다 교집합 크기를 센다. 10 중 몇 개가 같은가
sets = {
    "차수": {row["name"] for row in deg_rows},
    "PageRank": {row["name"] for row in pr_rows},
    "매개": {row["name"] for row in bc_rows},
}
for left, right in [("PageRank", "차수"), ("매개", "차수"), ("PageRank", "매개")]:
    print(f"{left:9} 대 {right:9} 겹침 {len(sets[left] & sets[right])}/10")

| 비교 | 겹침 |
|---|---|
| PageRank 대 차수 | **3/10** |
| 매개 대 차수 | **2/10** |
| PageRank 대 매개 | **5/10** |

**세 잣대가 서로 다른 것을 재고 있습니다.** 어느 하나가 나머지의 대체품이 아니라는 뜻입니다.

그러면 겹침이 크게 나오는 그래프는 어떨까요? 따라하기에서 쓴 `subwayGraph` 가 그렇습니다. 거기서는 **PageRank 대 차수가 9/10** 으로 거의 같습니다. 역 대부분이 앞뒤 역 둘하고만 이어진 사슬이라, "어떤 이웃인가"를 따져도 "몇 개인가"와 거의 같은 답이 나오는 것이죠.

그런데 **같은 그래프에서 매개 대 차수는 5/10** 입니다. 절반이 갈립니다. 여기서 읽어 낼 것은 이것입니다. 구조가 단순하다고 세 잣대가 **다 같아지는 것이 아닙니다.** **어떤 잣대는 겹치고 어떤 잣대는 갈립니다.** 어느 쪽인지는 그래프마다 다릅니다.

**그래서 매번 확인해야 합니다.** "중심성은 원래 다 다르다"고 외우는 것도, "단순한 그래프면 다 비슷하다"고 외우는 것도 아닙니다. 지금 이 그래프에서 어느 쌍이 겹치는지 세어 보는 것입니다.

## 6-2. 한 잣대만 더: 근접 중심성(closeness)

중심성은 지금까지 본 것만 있는 것이 아닙니다. 자주 쓰이는 것 하나만 더 봅니다. **근접 중심성**은 **다른 노드들까지의 평균 거리가 짧은 자리**를 찾습니다. 매개 중심성이 **길목**을 찾는다면 근접 중심성은 **어디로든 빨리 닿는 자리**를 찾습니다.

**여기서 "거리" 는 홉 수입니다.** 선을 몇 번 타고 가야 닿는지를 셉니다. 오늘 투영에는 가중치를 주지 않았으니 **선 하나가 곧 1** 이고, 두 노드 사이의 거리는 최단 경로의 선 개수입니다.

| 용어 | 이 알고리즘에서의 뜻 |
|---|---|
| 거리 | 그 노드까지 가는 **최단 경로의 홉 수**. 직접 이웃이면 1, 한 다리 건너면 2 |
| 평균 거리 | 닿을 수 있는 노드 전부의 홉 수를 더해 **그 개수로 나눈 값** |
| 점수 | 그 **평균 거리의 역수**. 평균이 1 이면 `1.0`, 평균이 4 면 `0.25` |

역수이므로 **점수가 클수록 가깝습니다.** 그리고 나올 수 있는 가장 큰 값은 `1.0` 입니다. 닿는 노드가 전부 한 홉 거리일 때죠. 이 값을 기억해 두세요.

그런데 이 알고리즘에는 함정이 하나 있습니다. 그냥 돌려 보면 바로 드러납니다.

In [ ]:
# 근접 중심성을 그대로 돌려 본다. 점수가 높을수록 '평균적으로 가깝다'는 뜻이다
close_rows = run_cypher("""
    CALL gds.closeness.stream('drugGraph')  // 닿을 수 있는 노드까지의 평균 홉 수로 점수를 낸다
    YIELD nodeId, score
    RETURN gds.util.asNode(nodeId).name AS name, score
    ORDER BY score DESC, name
    LIMIT 5
""")
for rank, row in enumerate(close_rows, 1):
    print(f"{rank}. {row['name']:26} {row['score']:.4f}")

상위가 전부 **1.0000** 입니다. 만점이죠. 게다가 앞의 세 잣대 어디에도 없던 이름들입니다. 정말 이 노드들이 그래프의 한복판에 있는 걸까요? 이웃을 세어 보면 압니다.

In [ ]:
# 1위로 나온 노드의 이웃 수를 센다. 정말 '어디로든 가까운' 노드인지 확인하는 것이다
# drugGraph 에 담은 관계 5종만 세어야 투영 기준의 이웃 수가 된다
top_close = close_rows[0]["name"]
neighbors = run_cypher("""
    MATCH (n {name: $name})-[r]-()  // 레이블을 안 적었다. 약효분류일 수도 약물일 수도 있어서다
    WHERE type(r) IN ['TREATS', 'PALLIATES', 'INCLUDES', 'RESEMBLES_DD', 'RESEMBLES_CC']
    RETURN count(*) AS cnt
""", name=top_close)[0]["cnt"]
print(f"{top_close} 의 이웃 수: {neighbors}")

이웃이 **하나**뿐입니다. 1위 `Amphenicols` 는 짝 하나(`Chloramphenicol`)하고만 이어져 있고 그 짝도 마찬가지라, 나머지 그래프로 나가는 선이 하나도 없습니다. 이 노드에서 출발해 닿을 수 있는 노드는 짝 하나가 전부이고, 나머지 2,010개에는 아예 닿지 못합니다.

근접 중심성은 **닿을 수 있는 노드까지의 거리만** 잽니다. 그러니 이 둘에게는 잴 거리가 **한 걸음 하나뿐**이고, 평균 거리가 1 이라 만점이 됩니다. 닿지 못하는 나머지는 계산에 들어가지도 않습니다. **가장 좁은 자리가 가장 높은 점수를 받은 것입니다.**

**에러는 나지 않습니다.** 오늘 여러 번 만난 그 모양이죠. 그래서 이럴 때는 **조화 근접 중심성(harmonic closeness)** 을 씁니다. 닿지 못하는 노드를 거리 무한대로 보고 그 몫을 0 으로 더하므로, 이렇게 끊겨 나온 짝은 만점을 받지 못합니다.

In [ ]:
# 알고리즘 이름에 .harmonic 만 붙였다. 앞 셀과 쿼리 모양이 같다. 점수도 클수록 가깝고 최대는 같은 1.0 이다
harmonic_rows = run_cypher("""
    CALL gds.closeness.harmonic.stream('drugGraph')  // 이름에 harmonic 만 붙었다. 닿지 못하는 노드를 0 으로 더한다
    YIELD nodeId, score
    RETURN gds.util.asNode(nodeId).name AS name, score
    ORDER BY score DESC, name
    LIMIT 10
""")
for rank, row in enumerate(harmonic_rows[:5], 1):
    print(f"{rank}. {row['name']:26} {row['score']:.4f}")

In [ ]:
# 새 잣대를 배웠으면 물을 것은 하나다. 이미 있는 잣대와 다른 답을 주는가
close_set = {row["name"] for row in harmonic_rows}
for label, other in [("차수", deg_rows), ("PageRank", pr_rows), ("매개", bc_rows)]:
    other_set = {row['name'] for row in other}
    print(f"근접(harmonic) 대 {label:9} 겹침 {len(close_set & other_set)}/10")


차수와 **5/10** 이 겹칩니다. 세 잣대끼리의 겹침(3/10 · 2/10 · 5/10)과 같은 수준이라 근접 중심성도 **제 나름의 답을 냅니다.** 6-1 의 비교표에 넣지 않은 것은 그 표가 오늘의 결론인 세 잣대만 견주는 자리이기 때문입니다. 다음의 고르는 기준 표에는 함께 넣었습니다.

> **잣대를 하나 더 배울 때마다 물을 것**: 이 잣대가 이미 쓰는 잣대와 **다른 답을 주는가.** 다르지 않다면 굳이 늘릴 이유가 없습니다. 그리고 잣대마다 **조용히 틀리는 자리**가 따로 있으니, 새 잣대를 쓸 때는 상위 몇 개를 직접 뜯어보는 습관이 필요합니다.

### 무엇을 알고 싶은가에 따라 고릅니다

| 알고 싶은 것 | 쓰는 중심성 | 무엇을 세는가 | 이 데이터에서의 1위 |
|---|---|---|---|
| 선이 가장 많은 노드 | 차수 | 이웃의 **개수** | hypertension |
| 점수가 큰 것들과 이어진 노드 | PageRank | 이웃이 넘겨 준 **몫의 합** | hypertension |
| 특정 노드 관점에서 **관련 깊은 것** | 개인화 PageRank | 한 출발점에서 **지나치는 빈도** | 1위는 출발점 자신. Sildenafil 기준 그다음이 Udenafil·Vardenafil |
| 무리를 잇는 길목 | 매개 중심성 | 그 노드를 **지나는 최단 경로 수** | hypertension (2위 Phenacemide) |
| 어디로든 빨리 닿는 자리 | 근접 중심성(`harmonic`) | 나머지까지 **홉 수의 역수를 더해 평균낸 값**(닿지 못하면 0 을 더한다) | Prednisone |

**고를 때 물어볼 것 네 가지**

1. **질문에 출발점이 있는가.** "이것과 비슷한 것"처럼 기준 노드가 문장에 들어 있으면 개인화 PageRank 입니다. 전체 순위로는 답이 안 나옵니다.
2. **이웃의 개수로 충분한가, 누구인지가 중요한가.** 개수면 차수, 누구인지까지면 PageRank 입니다. 오늘 `Eltrombopag` 가 차수 469위인데 PageRank 4위였던 것이 그 차이입니다.
3. **끊어지면 곤란한 자리를 찾는가.** 그러면 매개 중심성입니다. 선이 적어도 서로 다른 무리를 잇고 있으면 높게 나옵니다.
4. **평균적으로 가까운 자리를 찾는가.** 그러면 근접 중심성입니다. 다만 그래프가 끊겨 있으면 `gds.closeness` 는 끊겨 나온 조각에 만점을 주므로 **`gds.closeness.harmonic` 을 쓰세요.**

---
## 🚀 응용 클론코딩: 중심성 비교표 함수 만들기

오늘 한 일을 함수 하나로 묶습니다. **투영 이름과 개수를 받아 네 중심성 상위 목록을 한 표로 돌려주는** `centrality_report(graph_name, top_k)` 를 완성하세요.

1. `degree`·`pageRank`·`betweenness`·`closeness.harmonic` 을 각각 `stream` 으로 돌려 상위 `top_k` 개 이름을 모읍니다. 마지막 것은 이름에 점이 하나 더 들어갈 뿐 호출 모양은 같습니다.
2. 네 목록을 열로 갖는 DataFrame 을 만들어 돌려줍니다. 인덱스는 1부터 시작하는 순위입니다.
3. 만든 뒤 `drugGraph` 로 한 번, `subwayGraph` 로 한 번 불러 표를 확인하세요.

(1절의 `top5` 를 개수까지 받도록 바꾸면 그대로 재사용할 수 있습니다.)

**확인 기준**: 전철 쪽 근접 중심성 1위는 `서울역` 입니다. `harmonic` 을 쓰기 때문에 나온 값입니다. 그냥 `gds.closeness` 로 재면 본선과 끊긴 인천공항 세 역(`제1터미널`·`탑승동`·`제2터미널`)만 자기들끼리 이어져 있어서, 그중 **가운데인 `탑승동` 하나만** 양옆이 다 한 걸음이라 만점 `1.0` 을 받아 1위가 됩니다. 양끝 두 역은 평균 1.5 걸음이라 `0.667` 입니다. 6-2 에서 본 함정이 이 그래프에도 그대로 있습니다.

In [ ]:
# 🚀 응용 (아래 순서대로 직접 작성해 보세요)
# 1) centrality_report(graph_name, top_k) 를 정의한다
# 2) 알고리즘 이름 네 개를 리스트로 두고 반복하며 stream 을 돌린다
#    (degree · pageRank · betweenness · closeness.harmonic)
#    (f-string 으로 gds.{algo}.stream 을 만들고, 투영 이름과 개수는 파라미터로 넘긴다)
# 3) {한글 열이름: 이름목록} 형태의 dict 를 만들어 pd.DataFrame 으로 바꾼다
# 4) 인덱스를 1부터 top_k 까지로 바꿔 돌려준다
# 5) drugGraph 와 subwayGraph 로 각각 불러 display 한다

---
## 🧹 다 쓴 투영 내리기

지난 시간에 배운 대로 마무리합니다. 오늘은 투영을 셋 만들었고(`drugGraph`·`subwayGraph`·`fullGraph`), `fullGraph` 하나가 가장 큽니다(얼마나 되는지는 다음 셀이 찍어 줍니다). 그대로 두면 **서버 메모리에 그대로 남습니다.**

In [ ]:
# 내리기 전에 무엇이 얼마나 차지하는지 본다
for row in run_cypher("""
    CALL gds.graph.list()
    YIELD graphName, nodeCount, memoryUsage
    RETURN graphName, nodeCount, memoryUsage ORDER BY graphName
"""):
    print(f"  {row['graphName']:16} 노드 {row['nodeCount']:>6,}  {row['memoryUsage']}")

In [ ]:
# 오늘 만든 것만 이름으로 내린다. 카탈로그 전체를 지우면 남이 쓰는 투영까지 사라진다
for _name in ["drugGraph", "subwayGraph", "fullGraph"]:
    run_cypher("CALL gds.graph.drop($name, false) YIELD graphName RETURN graphName",
               name=_name)
print("남은 투영:", [r["graphName"] for r in
      run_cypher("CALL gds.graph.list() YIELD graphName RETURN graphName")])

> `mutate` 로 투영에 쌓아 둔 `deg`·`pr`·`bc` 도 이 순간 함께 사라집니다. 남겨야 할 값이 있다면 **내리기 전에** `nodeProperties.write` 로 원본에 넘겨야 합니다. 2절에서 본 순서가 이래서 중요합니다.

---
## 이번 강의 정리

| 개념 | 핵심 |
|---|---|
| 차수 | 선의 **개수**만 센다. `gds.degree.stream` |
| PageRank | 이웃의 **점수**까지 본다. `gds.pageRank.stream`. 점수는 상대값이라 순위로 읽는다 |
| PageRank 설정 | `dampingFactor` 0.85 · `maxIterations` 20 · `tolerance`. 기본 20회는 **수렴 보장이 아니다** |
| 실행 모드 | `stream`(화면) · `stats`(요약) · `mutate`(투영 안) · `write`(원본 속성) |
| 개인화 PageRank | `sourceNodes` 로 **출발점**을 정한다. 출발점이 대체로 1위 |
| 투영이 곧 질문 | 담지 않은 관계는 없는 것과 같다. 질문이 달라지면 투영도 달라진다 |
| 방향 함정 | 무방향으로 안 담으면 들어오는 선이 없는 종류가 통째로 바닥값에 동점으로 붙는다. 에러는 안 난다 |
| 검산 | 투영 관계 수가 Cypher 로 센 건수의 **2배**인지 본다 |
| 필터 | `nodeLabels`·`relationshipTypes` 는 결과가 아니라 **그래프를 줄인다** |
| 매개 중심성 | 최단 경로가 지나가는 **길목**. 선이 적어도 높을 수 있다 |
| 근접 중심성 | 평균 거리가 짧은 자리. 그래프가 끊겨 있으면 `gds.closeness` 는 **끊겨 나온 조각**에 만점을 준다(`harmonic` 을 쓴다) |
| 세 잣대 비교 | 상위 10 겹침이 3/10 · 2/10 · 5/10. **서로 다른 것을 잰다** |

**오늘 얻은 관점 하나**: 같은 그래프라도 **무엇을 담고 어떤 잣대로 재느냐**에 따라 답이 바뀝니다. 그래서 결과를 보고 "이 답이 어느 투영에서 어느 잣대로 나온 값인가"를 늘 함께 적어야 합니다.

## ⏭️ 예고: 다음 단원(34일차)

**커뮤니티 탐지와 유사도, 경로 탐색**을 다룹니다. 오늘은 노드 하나하나에 점수를 매겼다면, 다음에는 **노드들을 묶어 무리를 찾고**, 두 노드가 얼마나 닮았는지 재고, 둘 사이의 길을 찾습니다.